In [1]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  결과기 1번 셀 — 팀 구현
# =====================================================================================
#  ┌─ 유지하는 계약 ────────────────────────────────────────────────────────────────┐
#  │ · answer_question(question: str) 함수 이름과 입력 형식                         │
#  │ · 반환값: {"answer": 문자열, "retrieved": [[문서명, 조번호], ...]} 1~4개        │
#  │ · 전역 FastAPI app, GET /health, POST /answer                                 │
#  │ · Qwen2.5-Instruct 계열 생성 모델을 Colab T4 에서 로컬 실행                    │
#  │ · 새 Colab T4 런타임에서 위에서 아래로 한 번 실행하면 끝난다                   │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  설계
#    · 약관 4종 72개 조를 이 셀에 문자열로 담는다. 채점 당일 kakao.com 이 막히거나
#      원문이 바뀌어도 결과가 흔들리지 않는다. 드라이브·업로드·크롤링을 쓰지 않는다.
#    · 청크는 조(條) 단위다. retrieved 계약이 [문서명, 조번호] 라 조보다 잘게 쪼개면
#      근거를 되짚을 수 없고, 더 크게 묶으면 열거형 문항에서 뒷부분을 잃는다.
#    · 검색은 BM25(어절+음절 bigram) 와 임베딩을 순위 융합(RRF)한다. 질문이 약관
#      이름을 직접 부르면 그 약관 쪽에 가산점을 준다.
#    · 프롬프트에 넣는 조항과 retrieved 에 적는 조항을 같게 맞춘다.
#    · 문항 수·문항 번호를 코드에 넣지 않는다. 질문 문자열만 받아 처리한다.
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역
# =====================================================================================
import json
import math
import os
import re
import subprocess
import sys
import unicodedata
from collections import Counter

# torch 를 import 하기 전에 잡아야 효과가 있다. 긴 프롬프트에서 생기는 메모리
# 단편화로 OOM 이 나는 것을 줄인다.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

# True: 7B 4bit (품질) / False: 3B fp16 (안정). T4 는 Turing 이라 bf16 을 못 쓴다.
USE_7B = True

if USE_7B:
    # 4bit 양자화에만 필요하다. -U 를 붙이지 않으므로 torch 를 건드리지 않는다.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=False)

from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" if USE_7B else "Qwen/Qwen2.5-3B-Instruct"
EMB_MODEL_ID = "BAAI/bge-m3"          # 런타임 안에서 로컬 실행. 원격 API 가 아니다.

TOP_K = 4                 # retrieved 계약 상한. MRR 은 첫 적중 순위만 보므로 꽉 채운다.
DOC_NAME_BOOST = 0.5      # 질문이 약관 이름을 부를 때 그 약관에 주는 가산점 비율
SEED = 42                 # 컨텍스트 예산과 생성 상한은 1-4 답변 조립부에 있다

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


# ----- 1-1. 약관 원문 4종 72조 -------------------------------------------------------
# 출처: 카카오계정 약관(2026-05-29), 카카오 위치정보 이용약관(2026-07-16),
#       카카오 통합서비스약관(2026-05-29), 카카오 통합 약관(2022-08-25 아카이브).
ARTICLES = json.loads(r'''[{"document":"카카오계정 약관","article":1,"title":"목적","text":"주식회사 카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 다양한 인터넷과 모바일 서비스를 좀 더 편리하게 이용할 수 있도록 회사 또는 관계사의 개별 서비스에 모두 접속 가능한 통합로그인계정 체계를 만들고 그에 적용되는 '카카오계정 약관(이하 '본 약관')을 마련하였습니다. 본 약관은 여러분이 카카오계정 서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":2,"title":"약관의 효력 및 변경","text":"①본 약관의 내용은 카카오계정 웹사이트 또는 개별 서비스의 화면에 게시하거나 기타의 방법으로 공지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다. ②회사는 필요한 경우 관련법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게 여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다. ③회사가 전항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 여러분이 개정약관에 동의하지 않을 경우 여러분은 이용계약을 해지할 수 있습니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":3,"title":"약관 외 준칙","text":"본 약관에 규정되지 않은 사항에 대해서는 관련법령 또는 회사가 정한 개별 서비스의 이용약관, 운영정책 및 규칙 등(이하 ‘세부지침’)의 규정에 따릅니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":4,"title":"용어의 정의","text":"①본 약관에서 사용하는 용어의 정의는 다음과 같습니다. 1.카카오계정: 회사 또는 관계사가 제공하는 개별 서비스를 하나의 로그인계정과 비밀번호로 회원 인증, 회원정보 변경, 회원 가입 및 탈퇴 등을 관리할 수 있도록 회사가 정한 로그인계정 정책을 말합니다. 2.회원: 카카오계정이 적용된 개별 서비스 또는 카카오계정 웹사이트에서 본 약관에 동의하고, 카카오계정을 이용하는 자를 말합니다. 3.관계사: 회사와 제휴 관계를 맺고 카카오계정을 공동 제공하기로 합의한 법인을 말합니다. 개별 관계사는 카카오 기업사이트에서 확인할 수 있고 추후 추가/변동될 수 있으며 관계사가 추가/변동될 때에는 카카오 기업사이트에 변경 사항을 게시합니다. 4.개별 서비스: 카카오계정을 이용하여 접속 가능한 회사 또는 관계사가 제공하는 서비스를 말합니다. 개별 서비스는 추후 추가/변동될 수 있으며 서비스가 추가/변동될 때에는 카카오 기업사이트에 변경 사항을 게시합니다. 5.카카오계정 웹사이트: 회원이 온라인을 통해 카카오계정 정보를 조회 및 수정할 수 있는 인터넷 사이트를 말합니다. 6.카카오계정 정보 : 카카오계정을 이용하기 위해 회사가 정한 필수 내지 선택 입력 정보로서 카카오계정 웹사이트 또는 개별 서비스 내 카카오계정 설정 화면을 통해 정보 확인, 변경 처리 등을 관리할 수 있는 회원정보 항목을 말합니다. 7.이용기관 : 제11조에 따른 디지털카드서비스와 관련하여 디지털카드를 제출받거나 확인하여 자신의 업무, 영업에 활용하는 제3자를 말합니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":5,"title":"계약의 성립","text":"①카카오계정 이용 신청은 개별 서비스 또는 카카오계정 웹사이트 회원가입 화면에서 여러분이 카카오계정 정보에 일정 정보를 입력하는 방식으로 이루어집니다. ②카카오계정 이용계약은 여러분이 본 약관의 내용에 동의한 후 본 조 제1항에서 정한 이용신청을 하면 회사가 입력된 일정 정보를 인증한 후 가입을 승낙함으로써 체결됩니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":6,"title":"카카오계정 이용의 제한","text":"①제5조에 따른 가입 신청자에게 회사는 원칙적으로 카카오계정의 이용을 승낙합니다. 다만, 회사는 아래 각 호의 경우에는 그 사유가 해소될 때까지 승낙을 유보하거나 승낙하지 않을 수 있습니다. 특히, 여러분이 만 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 카카오계정을 생성할 수 있습니다. 1.회사가 본 약관 또는 세부지침에 의해 여러분의 카카오계정을 삭제하였던 경우 2.여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 카카오계정을 생성하려 한 경우 3.카카오계정 생성 시 필요한 정보를 입력하지 않거나 허위의 정보를 입력한 경우 4.제공 서비스 설비 용량에 현실적인 여유가 없는 경우 5.서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우 6.기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우 7.회사로부터 회원자격정지 조치 등을 받은 회원이 그 조치기간에 이용계약을 임의로 해지하고 재이용을 신청하는 경우 8.기타 관련법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우 ②만약, 여러분이 위 조건에 위반하여 카카오계정을 생성한 것으로 판명된 때에는 회사는 즉시 여러분의 카카오계정 이용을 정지시키거나 카카오계정을 삭제하는 등 적절한 제한을 할 수 있습니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":7,"title":"카카오계정 제공","text":"①회사가 개별 서비스와 연동하여 카카오계정에서 제공하는 서비스(이하 “카카오계정 서비스” 또는 “서비스”) 내용은 아래와 같습니다. 1.통합로그인 : 카카오계정이 적용된 개별 서비스에서 하나의 카카오계정과 비밀번호로 로그인할 수 있는 통합 회원 인증 서비스를 이용할 수 있습니다. 2.SSO(Single Sign On): 웹브라우저나 특정 모바일 기기에서 카카오계정 1회 로그인으로 여러분이 이용 중인 개별 서비스간 추가 로그인 없이 자동 접속 서비스를 이용할 수 있습니다. 3.카카오계정 정보 통합 관리 : 개별 서비스 이용을 위해 카카오계정 정보를 통합 관리합니다. 또한, 여러분이 이용하고자 하는 개별 서비스의 유형에 따라 전문기관을 통한 실명확인 및 본인인증을 요청할 수 있고, 이를 카카오계정 정보로 저장합니다. 4.사업자/단체 카카오계정 : 사업자/단체 명의로 카카오 서비스를 이용하기 위해 만들어진 카카오계정으로서 해당 사업자/단체의 책임 하에 권한을 위임받은 담당자가 이용, 관리할 수 있는 계정 서비스입니다. 5.기타 회사가 제공하는 서비스 ②회사는 더 나은 카카오계정 서비스의 제공을 위하여 여러분에게 서비스의 이용과 관련된 각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 서비스화면 내에 표시하거나 여러분의 이메일로 전송할 수 있습니다. 광고성 정보 전송의 경우에는 사전에 수신에 동의한 경우에만 전송합니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":8,"title":"카카오계정 서비스의 변경 및 종료","text":"①회사는 카카오계정 서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 아래 각 호의 경우 카카오계정 서비스의 전부 또는 일부를 제한하거나 중지할 수 있습니다. 1.카카오계정 서비스용 설비의 유지·보수 등을 위한 정기 또는 임시 점검의 경우 2.정전, 제반 설비의 장애 또는 이용량의 폭주 등으로 정상적인 카카오계정 이용에 지장이 있는 경우 3.관계사와의 계약 종료, 정부의 명령/규제, 서비스/회원 정책 변경 등 회사의 제반 사정으로 카카오계정 서비스를 유지할 수 없는 경우 4.기타 천재지변, 국가비상사태 등 불가항력적 사유가 있는 경우 ②전항에 의한 카카오계정 서비스 중단의 경우에는 미리 제14조에서 정한 방법으로 여러분에게 통지 내지 공지하겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다. 다만, 회사로서도 예측할 수 없거나 통제할 수 없는 사유(회사의 과실이 없는 디스크 내지 서버 장애, 시스템 다운 등)로 서비스가 중단된 경우에는 사전 통지 내지 공지를 할 수 없습니다. 이러한 경우에도 회사가 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하되, 2시간 이상 복구가 지연될 시 카카오 서비스 공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":9,"title":"카카오계정 관리","text":"①카카오계정은 여러분 본인만 이용할 수 있으며, 다른 사람이 여러분의 카카오계정을 이용하도록 허락할 수 없습니다. 그리고 여러분은 다른 사람이 여러분의 카카오계정을 무단으로 사용할 수 없도록 직접 비밀번호를 관리하여야 합니다. 회사는 다른 사람이 여러분의 카카오계정을 무단으로 사용하는 것을 막기 위하여 비밀번호 입력 및 추가적인 본인 확인 절차를 거치도록 할 수 있습니다. 만약 무단 사용이 발견된다면, 고객센터를 통하여 회사에게 알려주시기 바라며, 회사는 무단 사용을 막기 위한 방법을 여러분에게 안내하도록 하겠습니다. ②여러분은 카카오계정 웹사이트 또는 개별 서비스 내 카카오계정 설정 화면을 통하여 여러분의 카카오계정 정보를 열람하고 수정할 수 있습니다. 다만, 카카오계정 서비스의 제공 및 관리를 위해 필요한 카카오계정, 전화번호, 단말기 식별번호, 기타 본인확인정보 등 일부 정보는 수정이 불가능할 수 있으며, 수정하는 경우에는 추가적인 본인 확인 절차가 필요할 수 있습니다. ③여러분이 이용 신청 시 알려주신 내용에 변동이 있을 때, 전항에 따라 직접 수정하시거나, 고객센터를 통하여 회사에 알려 주시기 바랍니다. 여러분이 카카오계정 정보를 적시에 수정하지 않아 발생하는 문제에 대하여 회사의 고의 또는 과실이 없는 한 회사는 책임을 부담하지 아니합니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":10,"title":"사업자/단체 카카오계정","text":"①사업자/단체 카카오계정은 사업자등록번호 또는 고유번호가 있는 사업자/단체가 권한을 위임받은 담당자(이하 본조에서 ‘담당자’)를 통해 만들어 이용할 수 있습니다. 사업자/단체 카카오계정의 이용 및 관리에 관한 책임은 해당 사업자/단체에 있으며, 회사는 이와 관련한 책임을 지지 않습니다. ②사업자/단체 카카오계정은 계정 정보에 등록된 담당자 1인만 이용할 수 있으며, 이를 다른 사람에게 공유하는 것은 금지됩니다. ③사업자/단체 카카오계정은 개인 카카오계정으로 전환할 수 없고, 다른 개인 또는 법인 등 제3자에게 양도할 수 없습니다. ④사업자/단체 카카오계정은 일부 카카오 서비스의 가입 및 이용이 제한되며, 가입 및 이용이 제한되는 서비스는 정책에 따라 변경될 수 있습니다. ⑤사업자/단체 카카오계정의 정보 변경 또는 담당자 변경 요청에 대해 회사는 해당 계정에 대한 정당한 권한이 있는지 확인하기 위하여 일정한 증빙서류를 요청할 수 있습니다. ⑥사업자/단체 카카오계정은 사업자/단체에 귀속되는 것으로, 담당자는 해당 계정에 대해 권리를 주장할 수 없습니다. ⑦본 조에서 정하고 있는 내용 외에 사업자/단체 카카오계정과 관련된 상세한 사항은 사업자/단체 카카오계정 운영정책에 따르며, 회사는 게시판 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":11,"title":"디지털카드 서비스","text":"①회사는 회사를 포함한 제휴 발급기관의 요청에 따라 회원의 카카오계정에 자격증명, 티켓, 아이템 등의 디지털카드를 발급하고 이를 제휴 이용기관에 제출하는 등의 활용을 할 수 있도록 하는 서비스(이하 ‘디지털카드 서비스’라 합니다)를 제공합니다. ②디지털카드는 회원 본인의 신청에 따라 발급되거나, 이용자가 일정한 조건을 충족한 경우 또는 발급기관의 요청을 받은 경우에는 자동으로 발급될 수 있습니다. 단, 회사는 디지털카드 발급 과정에서 추가적인 인증 또는 이용 등의 동의를 요청할 수 있고, 해외 거주 또는 외국인 회원의 경우 디지털카드의 발급 및 이용이 제한 될 수 있습니다. ③회사는 제휴 발급기관이 제공하는 정보를 디지털카드에 담거나 표시할 뿐, 제휴 발급기관이 발급한 디지털카드의 내용에 대한 검증 및 적법성 등에 대한 보증을 하지 않습니다. ④회사, 발급기관 또는 이용기관(이하 발급기관과 이용기관을 통칭하여 ‘제휴사’라 합니다)이 제공하는 서비스에서 회원의 디지털카드 정보를 조회하거나 표시, 노출할 수 있습니다. ⑤디지털카드는 발급기관의 필요와 요청에 따라 회수 또는 수정될 수 있습니다. 회수된 디지털카드 및 디지털카드에 담긴 정보는 복구할 수 없습니다. ⑥디지털카드에는 발급기관이 설정한 유효기간이 있으며, 유효기간이 경과하는 경우 제휴사의 정책에 따라 디지털카드의 기능을 사용할 수 없거나 기타 제출 등의 활용에 제한이 있을 수 있습니다. 또한, 디지털카드의 활용처는 제휴사의 사정에 따라 변동될 수 있고, 회사는 디지털카드의 영속성을 보장하지 않으며, 기능 외의 금전적 가치도 인정하지 않습니다. ⑦회원이 카카오계정을 탈퇴하는 경우 해당 카카오계정에 발급되어 있는 디지털카드는 삭제되고, 동일한 디지털카드의 재발급이 불가능할 수 있습니다. ⑧회원은 디지털카드 서비스를 이용함에 있어서 아래 각 호의 행위는 하여서는 안 됩니다. 1.서비스 이용 시 허위 사실을 기재하거나, 타인의 명의 및 정보를 도용하여 회사가 제공하는 서비스 또는 디지털카드를 이용하는 행위 2.디지털카드 정보를 회원 본인이 아닌 제3자가 사용하도록 대여하는 행위 3.유효하지 않은 디지털카드를 비정상적 목적으로 사용하는 행위 4.서비스에서 회사가 게시한 정보의 무단 변경 또는 회사가 정한 정보 이외의 정보(컴퓨터 프로그램 등)등의 송신 또는 게시하는 행위 5.회사가 정하지 않은 비정상적인 방법으로 서비스를 이용하거나 시스템에 접근하는 행위 6.회사가 정하지 않은 비정상적인 방법으로 부당하게 디지털카드를 주고 받는 행위(예: 디지털카드의 유상거래, 이용자간 합의되지 않은 전송에 따른 탈취 행위, 정상적으로 안내되지 않은 방법에 의한 거래 행위 등) 7.기타 관련법령, 회사의 약관 및 운영정책을 위반하여 회사나 제휴사 또는 다른 제3자에게 손해를 끼치거나 손해를 끼칠 것으로 합리적으로 예상되는 경우 ⑨회사는 디지털카드의 활용과 관련하여 회원, 발급기관, 이용기관 간의 관계에서 어떠한 책임도 부담하지 않으며, 회사는 발급기관과 이용기관의 귀책사유로 인하여 회원에게 발생한 손해에 대하여 회사의 귀책사유가 없는 한 책임을 지지 않습니다. ⑩본 조에서 정하고 있는 내용 외에 디지털카드 서비스와 관련된 상세한 사항은 디지털카드 서비스 운영정책에 따르며, 회사는 서비스 공지사항 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":12,"title":"회원의 의무","text":"①여러분이 카카오계정 서비스를 이용할 때 아래 각 호의 행위는 하여서는 안 됩니다. 1.이용 신청 또는 변경 시 허위 사실을 기재하거나, 다른 회원의 카카오계정 및 비밀번호를 도용, 부정하게 사용하거나, 다른 사람의 명의를 사용하거나 명의자의 허락 없이 문자메시지(SMS) 인증 등을 수행하는 행위 2.타인의 명예를 손상시키거나 불이익을 주는 행위 3.게시판 등에 음란물을 게재하거나 음란사이트를 연결(링크)하는 행위 4.회사 또는 제3자의 저작권 등 기타 권리를 침해하는 행위 5.공공질서 및 미풍양속에 위반되는 내용의 정보, 문장, 도형, 음성 등을 타인에게 유포하는 행위 6.카카오계정 서비스와 관련된 설비의 오동작이나 정보 등의 파괴 및 혼란을 유발시키는 컴퓨터 바이러스 감염 자료를 등록 또는 유포하는 행위 7.카카오계정 서비스의 운영을 고의로 방해하거나 안정적 운영을 방해할 수 있는 정보 및 수신자의 명시적인 수신거부의사에 반하여 광고성 정보 또는 스팸메일(Spam Mail)을 전송하는 행위 8.회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위 9.타인으로 가장하는 행위 및 타인과의 관계를 허위로 명시하는 행위 10.다른 회원의 개인정보를 수집, 저장, 공개하는 행위 11.자기 또는 타인에게 재산상의 이익을 주거나 타인에게 손해를 가할 목적으로 허위의 정보를 유통시키는 행위 12.윤락행위를 알선하거나 음행을 매개하는 내용의 정보를 유통시키는 행위 13.수치심이나 혐오감 또는 공포심을 일으키는 말이나 음향, 글이나 화상 또는 영상을 계속하여 상대방에게 도달하게 하여 상대방의 일상적 생활을 방해하는 행위 14.관련 법령에 의하여 그 전송 또는 게시가 금지되는 정보(컴퓨터 프로그램 포함)의 전송 또는 게시 행위 15.회사 또는 관계사의 직원이나 운영자를 가장하거나 사칭하여 또는 타인의 명의를 도용하여 글을 게시하거나 E-mail, 카카오톡 메시지 등을 발송하는 행위 16.컴퓨터 소프트웨어, 하드웨어, 전기통신 장비의 정상적인 가동을 방해, 파괴할 목적으로 고안된 소프트웨어 바이러스, 기타 다른 컴퓨터 코드, 파일, 프로그램을 포함하고 있는 자료를 게시하거나 E-mail, 카카오톡 메시지 등으로 발송하는 행위 17.기타 불법한 행위 ②여러분은 서비스의 이용권한, 기타 이용계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할 수 없습니다. ③혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행위 등을 조사할 수 있고, 여러분의 계정・서비스 이용을 잠시 또는 계속하여 중단하거나, 재가입에 제한을 둘 수도 있습니다. 또한 여러분이 서비스와 관련된 설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 서비스 제공에 악영향을 미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다. ④본 조에서 정한 사항 및 그 밖에 카카오계정 서비스의 이용에 관한 자세한 사항은 카카오 운영정책 등을 참고해 주시기 바랍니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":13,"title":"개인정보의 보호","text":"여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은 카카오 개인정보처리방침을 참고하여 주십시오.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":14,"title":"회원에 대한 통지 및 공지","text":"회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 고객센터에 방문하여 의견을 개진할 수 있습니다. 서비스 이용자 전체에 대한 공지는 칠(7)일 이상 서비스 공지사항란에 게시함으로써 효력이 발생합니다. 여러분에게 중대한 영향을 미치는 사항의 경우에는 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":15,"title":"이용계약 해지","text":"①여러분이 카카오계정 이용을 더 이상 원치 않는 때에는 언제든지 서비스 내 제공되는 메뉴를 이용하여 이용계약의 해지 신청을 할 수 있으며, 회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다. ②회사는 여러분이 카카오계정 서비스를 이용하기 위해 카카오계정 로그인 혹은 접속한 기록이 없는 경우 여러분이 등록한 이메일주소, 휴대폰번호로 이메일, 문자메시지 또는 카카오톡 메시지를 보내는 등 기타 유효한 수단으로 통지 후 여러분의 카카오계정 정보를 파기하거나 분리 보관할 수 있으며, 이로 인해 카카오계정 서비스 이용을 위한 필수적인 정보가 부족할 경우 이용계약이 해지될 수도 있습니다. 이와 관련된 보다 자세한 사항은 카카오 운영정책의 서비스 장기 미이용 처리 정책을 참고하시기 바랍니다. ③이용계약이 해지되면 법령 및 개인정보 처리방침에 따라 여러분의 정보를 보유하는 경우를 제외하고는 여러분의 카카오계정 정보 및 카카오계정으로 이용하였던 개별 서비스 데이터는 삭제됩니다. 다만, 여러분이 개별 서비스 내에서 작성한 게시물 등 모든 데이터의 삭제와 관련한 사항은 개별 서비스의 약관에 따릅니다. ④이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다.","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":16,"title":"손해배상","text":"①회사는 법령상 허용되는 한도 내에서 서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 회원이 작성하는 등의 방법으로 서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다. ②회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 관련 법령에 따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징벌적 손해에 대한 책임을 부담하지 않습니다. 1.천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해 2.여러분의 귀책사유로 서비스 이용에 장애가 발생한 경우 3.서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해 4.제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해 5.제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해 6.제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해 7.전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에서 발생된 손해 8.기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해","effective_date":"2026-05-29"},{"document":"카카오계정 약관","article":17,"title":"분쟁의 해결","text":"본 약관 또는 서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 서비스 이용과 관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사소송법상의 관할법원에 소를 제기할 수 있습니다. 공고일자 : 2026년 5월 13일 시행일자 : 2026년 5월 29일 변경 전 카카오계정 약관 보기 서비스 이용정보 이용약관 위치정보 이용약관 개인정보 처리방침 운영정책 청소년보호정책 브랜드보호정책 권리침해신고안내 공지사항 새로운 알림이 있습니다 Copyright © Kakao Corp. All rights reserved. 맨위로","effective_date":"2026-05-29"},{"document":"카카오 위치정보 이용약관","article":1,"title":"목적","text":"본 약관은 주식회사 카카오(이하 \"회사\")가 제공하는 사물위치정보 및 위치기반 서비스(이하, 위치정보 서비스)에 대해 회사와 서비스를 이용하는 이용자간의 권리·의무 및 책임사항, 기타 필요한 사항 규정을 목적으로 합니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":2,"title":"이용약관의 효력 및 변경","text":"①본 약관은 이용자가 본 약관에 동의하고 회사가 정한 절차에 따라 위치정보 서비스의 이용자로 등록됨으로써 효력이 발생합니다. ②이용자가 본 약관의 “동의하기” 버튼을 클릭하였을 경우 본 약관의 내용을 모두 읽고 이를 충분히 이해하였으며, 그 적용에 동의한 것으로 봅니다. ③회사는 위치정보 서비스의 변경사항을 반영하기 위한 목적 등으로 필요한 경우 관련 법령을 위배하지 않는 범위에서 본 약관을 수정할 수 있습니다. ④약관이 변경되는 경우 회사는 변경사항을 그 적용일자 최소 15일 전에 회사의 홈페이지 또는 서비스 공지사항 등(이하, 홈페이지 등)을 통해 공지합니다. 다만, 개정되는 내용이 이용자 권리의 중대한 변경을 발생시키는 경우 적용일 최소 30일 전에 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 발송하는 방법 등으로 개별적으로 고지합니다. ⑤회사가 전항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 이용자의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 이용자가 개정약관에 동의하지 않을 경우 본 약관에 대한 동의를 철회할 수 있습니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":3,"title":"약관 외 준칙","text":"이 약관에 명시되지 않은 사항에 대해서는 위치 정보의 보호 및 이용 등에 관한 법률, 개인정보보호법, 전기통신사업법, 정보통신망 이용촉진 및 정보보호 등에 관한 법률 등 관계법령 및 회사가 정한 지침 등의 규정에 따릅니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":4,"title":"서비스의 내용","text":"회사는 위치정보사업자로부터 수집한 이용자의 위치정보 또는 직접 수집한 사물위치정보를 이용하여 아래와 같은 위치정보 서비스를 제공합니다. ①검색결과 제공 및 콘텐츠 추천 : 이용자의 위치나 경로를 바탕으로 관련 정보나 콘텐츠를 검색하거나 추천해주는 서비스를 제공합니다. ②생활편의 서비스 제공 : 이용자의 위치에 따른 길찾기, 경로 또는 이동수단 추천, 경로 안내 및 알림 서비스를 제공합니다. ③위치 기반 콘텐츠 분류(Geo Tagging) : 이용자가 작성한 게시글, 사진, 영상 등에 위치정보를 저장하거나, 위치를 기반으로 콘텐츠를 분류하는 기능을 제공합니다. ④위치기반 소셜 서비스 제공 : 내 위치를 다른 이용자와 공유하거나 콘텐츠 남기기 등 인터랙션을 포함한 위치 서비스를 제공합니다. ⑤위치기반 광고 : 이용자의 위치정보를 활용한 광고성 정보 안내, 검색 및 디스플레이 광고소재 제공, 맞춤형 광고를 제공합니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":5,"title":"서비스 이용요금","text":"회사가 제공하는 위치정보 서비스는 무료입니다. 단, 무선 서비스 이용 시 발생하는 데이터 통신료는 별도이며, 이용자가 가입한 각 이동통신사의 정책에 따릅니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":6,"title":"서비스의 변경・제한・중지","text":"①회사는 정책변경 또는 관련법령 변경 등과 같은 제반 사정을 이유로 위치기반서비스를 유지할 수 없는 경우 위치기반서비스의 전부 또는 일부를 변경·제한·중지할 수 있습니다. ②회사는 아래 각호의 경우에는 이용자의 서비스 이용을 제한하거나 중지시킬 수 있습니다. 1.이용자가 회사 서비스의 운영을 고의 또는 중과실로 방해하는 경우 2.서비스용 설비 점검, 보수 또는 공사로 인하여 부득이한 경우 3.전기통신사업법에 규정된 기간통신사업자가 전기통신 서비스를 중지했을 경우 4.국가비상사태, 서비스 설비의 장애 또는 서비스 이용의 폭주 등으로 서비스 이용에 지장이 있는 때 5.기타 중대한 사유로 인하여 회사가 서비스 제공을 지속하는 것이 부적당하다고 인정하는 경우 ③회사가 제1항 및 제2항의 규정에 의하여 서비스 이용을 제한하거나 중지한 때에는 그 사유 및 제한기간 등을 회사 홈페이지 등을 통해 사전 공지하거나 이용자에게 통지합니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":7,"title":"개인위치정보의 이용 또는 제공","text":"①회사는 개인위치정보를 이용하여 위치기반서비스를 제공하는 경우 본 약관에 고지하고 동의를 받습니다. ②회사는 이용자의 동의 없이 개인위치정보를 제3자에게 제공하지 않으며, 제3자에게 제공하는 경우에는 제공받는 자 및 제공목적을 사전에 이용자에게 고지하고 동의를 받습니다. ③제2항에 따라 개인위치정보를 이용자가 지정하는 제3자에게 제공하는 경우 개인위치정보를 수집한 통신단말장치 또는 전자우편주소로 매회 이용자에게 제공받는 자, 제공일시 및 제공목적을 즉시 통지합니다. 단, 아래의 경우 이용자가 미리 특정하여 지정한 통신단말장치 또는 전자우편주소, 온라인게시 등으로 통지합니다. 1.개인위치정보를 수집한 당해 통신단말장치가 문자, 음성 또는 영상의 수신기능을 갖추지 아니한 경우 2.이용자의 개인위치정보를 수집한 통신단말장치 외의 통신단말장치 또는 전자우편주소, 온라인게시 등으로 통보할 것을 미리 요청한 경우 위치정보 제공 현황 자세히 보기","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":8,"title":"위치정보 수집·이용·제공사실 확인자료의 보관","text":"회사는 위치정보의 보호 및 이용 등에 관한 법률 제16조 제2항에 근거하여 위치정보 수집·이용·제공사실 확인자료를 위치정보시스템에 자동으로 기록·보존하며, 해당 자료는 6개월간 보관합니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":9,"title":"개인위치정보의 보유 목적 및 보유기간","text":"회사는 위치기반서비스 제공을 위해 아래와 같이 개인위치정보를 보유합니다. ①본 약관 제4조 따른 위치기반서비스 이용 및 제공 목적 달성한 때에는 지체없이 개인위치정보를 파기합니다. ②다만, 이용자가 작성한 게시물 또는 콘텐츠와 함께 위치정보가 저장되는 서비스의 경우 해당 게시물 또는 콘텐츠의 보관기간 동안 개인위치정보가 보관됩니다. ③그 외 위치기반서비스 제공을 위해 필요한 경우 이용목적 달성을 위해 필요한 최소한의 기간 동안 개인위치정보를 보유할 수 있습니다. ④위 1, 2, 3항에도 불구하고 다른 법령 또는 위치정보법에 따라 보유해야하는 정당한 사유가 있는 경우에는 그에 따릅니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":10,"title":"개인위치정보주체의 권리","text":"①이용자는 언제든지 개인위치정보를 이용한 위치기반서비스의 이용 및 제공에 대한 동의 전부 또는 일부를 유보할 수 있습니다. ②이용자는 언제든지 개인위치정보를 이용한 위치기반서비스의 이용 및 제공에 대한 동의 전부 또는 일부를 철회할 수 있습니다. 이 경우 회사는 지체 없이 철회된 범위의 개인위치정보 및 위치정보 이용·제공사실 확인자료를 파기합니다. ③이용자는 개인위치정보의 이용·제공의 일시적인 중지를 요구할 수 있습니다. 이 경우 회사는 이를 거절할 수 없으며 이를 충족하는 기술적 수단을 마련합니다 ④이용자는 회사에 대하여 아래 자료에 대한 열람 또는 고지를 요구할 수 있으며, 해당 자료에 오류가 있는 경우에는 정정을 요구할 수 있습니다. 이 경우 회사는 정당한 사유 없이 요구를 거절하지 않습니다. 1.이용자에 대한 위치정보 이용·제공사실 확인자료 2.이용자의 개인위치정보가 위치정보의 보호 및 이용 등에 관한 법률 또는 다른 법령의 규정에 의하여 제3자에게 제공된 이유 및 내용 ⑤이용자는 권리행사를 위해 본 약관 제14조의 연락처를 이용하여 회사에 요청할 수 있습니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":11,"title":"법정대리인의 권리","text":"①회사는 14세 미만의 이용자에 대해서는 개인위치정보를 이용한 위치기반서비스 제공 및 개인위치정보의 제3자 제공에 대한 동의를 이용자 및 이용자의 법정대리인으로부터 받아야 합니다. 이 경우 법정대리인은 본 약관 제10조에 의한 이용자의 권리를 모두 가집니다. ②회사는 14세 미만의 아동의 개인위치정보 또는 위치정보 이용, 제공사실 확인자료를 이용약관에 명시 또는 고지한 범위를 넘어 이용하거나 제3자에게 제공하고자 하는 경우 이용자와 이용자의 법정대리인의 동의를 받아야 합니다. 단, 아래의 경우는 제외합니다. 1.위치정보 및 위치기반서비스 제공에 따른 요금정산을 위하여 위치정보 이용, 제공사실 확인자료가 필요한 경우 2.통계작성, 학술연구 또는 시장조사를 위하여 특정 개인을 알아볼 수 없는 형태로 가공하여 제공하는 경우","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":12,"title":"8세 이하의 아동 등의 보호의무자의 권리","text":"①회사는 아래의 경우에 해당하는 자(이하 “8세 이하의 아동 등”)의 위치정보의 보호 및 이용 등에 관한 법률 제26조2항에 해당하는 자(이하 “보호의무자”)가 8세 이하의 아동 등의 생명 또는 신체보호를 위하여 개인위치정보의 이용 또는 제공에 동의하는 경우에는 본인의 동의가 있는 것으로 봅니다. 1.8세 이하의 아동 2.피성년후견인 3.장애인복지법 제2조제2항제2호에 따른 정신적 장애를 가진 사람으로서 장애인고용촉진 및 직업재활법 제2조제2호에 따른 중증장애인에 해당하는 사람(장애인복지법 제32조에 따라 장애인 등록을 한 사람만 해당한다) ②8세 이하의 아동 등의 생명 또는 신체의 보호를 위하여 개인위치정보의 이용 또는 제공에 동의를 하고자 하는 보호의무자는 서면동의서에 보호의무자임을 증명하는 서면을 첨부하여 회사에 제출하여야 합니다. ③보호의무자는 8세 이하의 아동 등의 개인위치정보 이용 또는 제공에 동의하는 경우 본 약관 제9조에 의한 이용자의 권리를 모두 가집니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":13,"title":"손해배상","text":"회사의 위치정보의 보호 및 이용 등에 관한 법률 제15조 및 26조의 규정을 위반한 행위로 인해 손해를 입은 경우 이용자는 회사에 손해배상을 청구할 수 있습니다. 회사는 고의, 과실이 없음을 입증하지 못하는 경우 책임을 면할 수 없습니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":14,"title":"면책","text":"①회사는 다음 각 호의 경우로 위치기반서비스를 제공할 수 없는 경우 이로 인하여 이용자에게 발생한 손해에 대해서는 회사의 고의 과실이 없는 한 책임을 부담하지 않습니다. 1.천재지변 또는 이에 준하는 불가항력의 상태가 있는 경우 2.위치기반서비스 제공을 위하여 회사와 서비스 제휴계약을 체결한 제3자의 고의적인 서비스 방해가 있는 경우 3.이용자의 귀책사유로 위치기반서비스 이용에 장애가 있는 경우 4.제1호 내지 제3호를 제외한 기타 회사의 고의·과실이 없는 사유로 인한 경우 ②회사는 위치기반서비스 및 위치기반서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며 이로 인해 발생한 이용자의 손해에 대하여는 회사의 고의 과실이 없는 한 책임을 부담하지 아니합니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":15,"title":"분쟁의 조정 및 기타","text":"①회사는 위치정보와 관련된 분쟁의 해결을 위해 이용자와 성실히 협의합니다. ②전항의 협의에서 분쟁이 해결되지 않은 경우, 회사와 이용자는 위치정보의 보호 및 이용 등에 관한 법률 제28조의 규정에 의해 방송통신위원회에 재정을 신청하거나, 개인정보보호법 제43조의 규정에 의해 개인정보 분쟁조정위원회에 조정을 신청할 수 있습니다.","effective_date":"2026-07-16"},{"document":"카카오 위치정보 이용약관","article":16,"title":"사업자 및 위치정보관리책임자 정보","text":"① 회사의 상호, 주소 및 연락처는 아래와 같습니다. 상호 : 주식회사 카카오 주소 : 제주특별자치도 제주시 첨단로 242 (영평동) 대표전화 : 1577-3754 (유료) ② 회사는 개인위치정보를 적절히 관리·보호하고, 이용자의 불만을 원활히 처리할 수 있도록 실질적인 책임을 질 수 있는 지위에 있는 자를 위치정보관리책임자로 지정해 운영하고 있습니다. 위치정보관리책임자는 위치기반서비스를 제공하거나 관리하는 부서의 부서장으로서 성명과 연락처는 아래와 같습니다. 성명 : 김연지 대표전화 : 1577-3754 (유료) 위치정보 전용문의 게시판 (바로가기) <시행일자> 공고일자 : 2026년 7월 2일 시행일자 : 2026년 7월 16일 변경 전 위치정보 이용약관 보기 서비스 이용정보 이용약관 위치정보 이용약관 개인정보 처리방침 운영정책 청소년보호정책 브랜드보호정책 권리침해신고안내 공지사항 새로운 알림이 있습니다 Copyright © Kakao Corp. All rights reserved. 맨위로","effective_date":"2026-07-16"},{"document":"카카오 통합서비스약관","article":18,"title":"분쟁의 해결","text":"본 약관 또는 통합서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 통합서비스 이용과 관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사소송법의 관할법원에 소를 제기할 수 있습니다. 공고일자 : 2026년 5월 13일 시행일자 : 2026년 5월 29일 변경 전 통합서비스 약관 보기 서비스 이용정보 이용약관 위치정보 이용약관 개인정보 처리방침 운영정책 청소년보호정책 브랜드보호정책 권리침해신고안내 공지사항 새로운 알림이 있습니다 Copyright © Kakao Corp. All rights reserved. 맨위로","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":1,"title":"목적 및 정의","text":"주식회사 카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 회사가 제공하는 다양한 인터넷과 모바일 서비스(이하 해당 서비스들을 모두 합하여 “통합서비스” 또는 “서비스”라 함)에 더 가깝고 편리하게 다가갈 수 있도록 ‘카카오 통합서비스약관’(이하 ‘본 약관’)을 마련하였습니다. 여러분은 본 약관에 동의함으로써 통합서비스에 가입하여 통합서비스를 이용할 수 있습니다. 단, 여러분은 회사가 아닌 계열사를 포함한 제3자가 제공하는 서비스 (예: ㈜카카오모빌리티가 제공하는 카카오 T 택시 서비스)에 가입되지는 않으며, 회사가 제공하는 유료서비스의 경우 여러분이 별도의 유료이용약관에 대한 동의한 때에 회사와 여러분 간의 유료서비스 이용계약이 성립합니다. 본 약관은 여러분이 통합서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다. 통합서비스: 회사가 제공하는 1) “카카오” 브랜드를 사용하는 서비스(예:카카오톡) 또는 2) 카카오계정으로 이용하는 서비스(예: 브런치) (단, 서비스 명칭에 ‘카카오’가 사용되더라도 회사가 아닌 카카오 계열사에서 제공하는 서비스 (예: 카카오 T택시 서비스)는 본 약관의 통합서비스에 포함되지 않습니다) 개별 서비스: 통합서비스를 구성하는 세부 하위 서비스를 의미하며, 예를 들어 각 통합서비스 내의 유료서비스, 카카오톡 서비스 등을 의미함","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":2,"title":"약관의 효력 및 변경","text":"①본 약관의 내용은 통합서비스의 화면에 게시하거나 기타의 방법으로 공지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다. ②회사는 필요한 경우 관련 법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게 여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다. ③회사가 전 항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. ④여러분은 변경된 약관에 대하여 거부의사를 표시함으로써 이용계약의 해지를 선택할 수 있습니다. ⑤본 약관은 여러분이 본 약관에 동의한 날로부터 본 약관 제13조에 따른 이용계약의 해지 시까지 적용하는 것을 원칙으로 합니다. 단, 본 약관의 일부 조항은 이용계약의 해지 후에도 유효하게 적용될 수 있습니다","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":3,"title":"약관 외 준칙","text":"본 약관에 규정되지 않은 사항에 대해서는 관련 법령 또는 통합서비스를 구성하는 개별 서비스의 이용약관, 운영정책 및 규칙, 카카오 운영정책 및 규칙 등(이하 총칭하여 '세부 지침')의 규정에 따릅니다. 세부지침은 본 약관과 더불어 이용계약의 일부를 구성합니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":4,"title":"계약의 성립","text":"①통합서비스에 가입하기 위해서는 카카오계정이 필요합니다. 카카오계정이 없으신 경우 카카오계정을 먼저 생성하시기 바랍니다. ②통합서비스 이용계약은 여러분이 본 약관의 내용에 동의한 후 회사가 여러분의 카카오계정 정보 등을 확인한 후 승낙함으로써 체결됩니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":5,"title":"통합서비스 가입의 제한","text":"①제4조에 따른 가입 신청자에게 회사는 원칙적으로 통합서비스 가입을 승낙합니다. 다만, 회사는 아래 각 호의 경우에는 그 사유가 해소될 때까지 승낙을 유보하거나 승낙하지 않을 수 있습니다. 특히, 여러분이 만 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 통합서비스에 가입할 수 있습니다. 1.여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 통합서비스에 가입하려고 한 경우 2.통합서비스 제공 설비 용량에 현실적인 여유가 없는 경우 3.통합서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우 4.기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우 5.회사로부터 통합서비스 이용정지 조치 등을 받은 자가 그 조치기간에 통합서비스 이용계약을 임의로 해지하고 재가입을 신청하는 경우 6.기타 관련 법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우 ②만약, 여러분이 위 조건에 위반하여 통합서비스에 가입한 것으로 판명된 때에는 회사는 즉시 여러분의 통합서비스 이용을 정지시키거나 카카오계정을 삭제하는 등 적절한 제한을 할 수 있습니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":6,"title":"다양한 서비스의 제공","text":"①통합서비스 이용계약이 성립되면, 여러분은 통합서비스를 구성하는 개별 서비스를 여러분이 원하는 때에 자유롭게 이용할 수 있습니다. ②다만, 통합서비스 내에서도 일부 개별 서비스의 경우 별도의 이용약관에 동의해야 이용이 가능하며 필요한 추가 정보를 기재하거나, 이메일 주소 승인 또는 문자메시지 인증, 인증서 발급 등 회사가 정한 인증 절차를 완료하여야 서비스 이용이 가능합니다. ③여러분은 통합서비스 가입 후에도 언제든지 통합서비스를 구성하는 개별 서비스 화면 또는 메뉴에서 제공하는 기능을 이용하여 해당 개별 서비스 의 이용을 종료할 수 있으며, 이 경우 관련 법령에서 정하는 바에 따라 일정기간 보관해야 하는 정보를 제외하고는 해당 서비스 이용기록, 여러분이 작성한 게시물 등 모든 데이터는 즉시 삭제 처리됩니다. 다만, 여러분이 작성한 게시물이 제3자에 의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제3자의 게시물에 댓글 등 게시물을 추가하는 등의 경우에는 해당 게시물 및 댓글은 삭제되지 않으므로 반드시 이용 종료 전에 삭제하시기 바라며, 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도 있으니 이 점 유의하여 주시기 바랍니다. 개별 서비스 이용 종료 시점에 향후 일정기간 해당 서비스의 재이용에 제한이 있을 수 있다는 별도 안내가 있는 경우 해당 안내에 따라 해당 서비스의 재이용에 일정한 시간적 제한이 있을 수 있는 점 또한 유의하여 주시기 바랍니다. ④전항에 따른 개별 서비스의 이용 종료는 해당 개별 서비스의 이용 종료만을 의미하며, 통합서비스를 구성하는 다른 서비스의 이용이 종료되지는 않습니다. 여러분이 통합서비스 전체의 이용을 종료하고 싶은 경우에는 본 약관 제13조에서 정한 바처럼 통합서비스 이용계약을 해지하여야 합니다. ⑤회사는 여러분에게 SNS, 게시판 서비스, 온라인 콘텐츠 제공 서비스, 위치기반서비스 등 여러분이 인터넷과 모바일로 즐길 수 있는 다양한 서비스를 제공합니다. 여러분은 스마트폰의 어플리케이션 스토어 등에서 서비스를 다운받아 설치하거나 직접 PC에 설치 혹은 웹페이지에 접속하여 서비스를 이용할 수 있습니다. 그런데 회사는 여러분이 원하는 다양한 서비스를 시시각각 제공하기 때문에 서비스의 자세한 내용은 별도로 알려드릴 수밖에 없습니다. 이러한 회사의 사정을 이해하여 주시길 바라며, 회사도 개별적인 서비스 이용방법을 어플리케이션 스토어와 각 서비스의 Q&A 센터, 해당 안내 및 고지사항에서 더 상세하게 안내하고 있으니 언제든지 확인하여 주시기 바랍니다. ⑥회사는 여러분이 통합서비스를 마음껏 이용할 수 있도록 이에 필요한 소프트웨어의 개인적이고 전 세계적이며 양도불가능하고 비독점적인 무상의 라이선스를 여러분에게 제공합니다. 단, 회사가 여러분에게 회사의 상표 및 로고를 사용할 권리를 부여하는 것은 아니라는 점은 잊지 말아주시기 바랍니다. ⑦회사가 여러분에게 제공하는 통합서비스에는 인공지능에 기반하여 운용되는 서비스가 포함될 수 있으며, 회사가 인공지능에 의하여 생성된 결과물을 제공할 경우에는 관련 법에 따라 고지 및 표시합니다. ⑧회사는 더 나은 통합서비스를 위하여 통합서비스에 필요한 소프트웨어의 업데이트 버전을 제공할 수 있습니다. 소프트웨어의 업데이트에는 중요한 기능의 추가 또는 불필요한 기능의 제거 등이 포함되어 있습니다. 여러분들도 통합서비스를 즐겁게 이용할 수 있도록 꾸준히 업데이트를 하여 주시기 바랍니다. ⑨회사는 스팸성 메일(피싱, 바이러스 유포, 개인정보 탈취 등 각종 불법 및 사행성 스팸을 의미합니다)로부터 이용자를 보호하기 위해 수발신 메일에 대한 스팸 대응 및 보안 조치를 합니다. 더불어 유관기관의 권고가 있거나 이용자 보호를 위하여 필요하다고 판단하는 경우 이에 따른 추가적인 스팸 운영정책 및 기능을 제공합니다. ⑩회사는 더 나은 통합서비스의 제공을 위하여 여러분에게 통합서비스의 이용과 관련된 각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 통합서비스 내에 표시하거나 여러분의 카카오계정 정보에 등록되어 있는 연락처로 직접 발송할 수 있습니다. 단, 광고성 정보 전송의 경우에는 사전에 수신에 동의한 경우에만 전송합니다. ⑪통합서비스 이용 중 시스템 오류 등 문제점을 발견하신다면 언제든지 고객센터로 알려주시기 바랍니다. ⑫여러분이 통합서비스를 이용하는 과정에서 Wi-Fi 무선인터넷을 사용하지 않고, 가입하신 이동통신사의 무선인터넷에 연결하여 이용하는 경우 이동통신사로부터 여러분에게 별도의 데이터 통신요금이 부과될 수 있는 점을 유의하여 주시기 바랍니다. 통합서비스 이용 과정에서 발생하는 데이터 통신요금은 여러분이 여러분의 비용과 책임 하에 이동통신사에 납부하셔야 합니다. 데이터 통신요금에 대한 자세한 안내는 여러분이 가입하신 이동통신사에 문의하시기 바랍니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":7,"title":"통합서비스의 변경 및 종료","text":"①회사는 통합서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 아래 각 호의 경우 통합서비스의 전부 또는 일부를 제한하거나 중지할 수 있습니다. 1.통합서비스용 설비의 유지·보수 등을 위한 정기 또는 임시 점검의 경우 2.정전, 제반 설비의 장애 또는 이용량의 폭주 등으로 정상적인 통합서비스 이용에 지장이 있는 경우 3.관계사와의 계약 종료, 정부의 명령/규제, 서비스/회원 정책 변경 등 회사의 제반 사정으로 통합서비스의 전부 또는 일부를 유지할 수 없는 경우 4.기타 천재지변, 국가비상사태 등 불가항력적 사유가 있는 경우 ②전항에 의한 통합서비스 중단의 경우에는 미리 제17조에서 정한 방법으로 여러분에게 통지 내지 공지하겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다. 다만, 회사로서도 예측할 수 없거나 통제할 수 없는 사유(회사의 과실이 없는 디스크 내지 서버 장애, 시스템 다운 등)로 인해 사전 통지 내지 공지가 불가능한 경우에는 그러하지 아니합니다. 이러한 경우에도 회사는 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하고, 2시간 이상 복구가 지연되는 경우 서비스 공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":8,"title":"게시물의 관리","text":"①여러분의 게시물이 정보통신망 이용촉진 및 정보보호 등에 관한 법률(이하 ‘정보통신망법’)및 저작권법 등 관련 법령에 위반되는 내용을 포함하는 경우, 권리자는 회사에 관련 법령이 정한 절차에 따라 해당 게시물의 게시중단 및 삭제 등을 요청할 수 있으며, 회사는 관련 법령에 따라 조치를 취합니다. ②회사는 권리자의 요청이 없는 경우라도 권리침해가 인정될 만한 사유가 있거나 기타 회사의 정책 및 관련 법령에 위반되는 경우에는 관련 법령에 따라 해당 게시물에 대해 임시조치 등을 취할 수 있습니다. ③위와 관련된 세부 절차는 정보통신망법 및 저작권법이 규정한 범위 내에서 회사가 정한 권리침해 신고 절차 에 따릅니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":9,"title":"권리의 귀속 및 저작물의 이용","text":"① 여러분은 사진, 글, 정보, (동)영상, 통합서비스 또는 회사에 대한 의견이나 제안 등 콘텐츠(이하 ‘게시물’)를 통합서비스 내에 게시할 수 있으며, 이러한 게시물에 대한 저작권을 포함한 지적재산권은 당연히 권리자가 계속하여 보유합니다. ② 여러분은 통합서비스 내에 게시한 게시물에 대한 사용, 저장, 수정, 복제, 공중송신, 전시, 배포 등의 방식으로 이용할 수 있도록 사용을 허락하는 전 세계적인 라이선스를 회사에게 제공하게 됩니다. 본 라이선스에서 여러분이 회사에게 부여하는 권리는 통합서비스를 운영, 개선, 홍보하고 새로운 서비스를 개발하기 위한 범위 내에서 사용되며, 이러한 목적 범위 내에서 회사와 명시적인 업무계약을 체결한 상대방 또는 다른 이용자에 대한 서브라이선스 또한 여기에 포함됩니다. 또한, 통합서비스의 개선 및 연구개발 목적으로 회사 및 회사의 계열사에서 게시물을 사용할 수 있습니다. 일부 개별 서비스에서는 여러분이 제공한 콘텐츠에 접근하거나 이를 삭제하는 방법을 제공할 수 있습니다(다만 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도 있습니다). 또한 일부 서비스에서는 제공된 콘텐츠에 대한 회사의 사용 범위를 제한하는 설정이 있습니다. ③여러분은 회사에 제공한 콘텐츠에 대해 회사에 라이선스를 부여하기 위해 필요한 권리를 보유해야 합니다. 이러한 권리를 보유하지 않아 발생하는 모든 문제에 대해서는 게시자가 책임을 부담하게 됩니다. 또한, 여러분은 음란하거나 폭력적이거나 기타 공서양속 및 법령에 위반하는 콘텐츠를 공개 또는 게시할 수 없습니다. ④회사는 여러분의 콘텐츠가 관련 법령에 위반되거나 음란 또는 청소년에게 유해한 게시물, 차별 갈등을 조장하는 게시물, 도배 · 광고 · 홍보 · 스팸성 게시물, 계정을 양도 또는 거래하는 내용의 게시물, 타인을 사칭하는 게시물 등이라고 판단되는 경우 이를 삭제하거나 게시를 거부할 수 있습니다. 다만 회사가 모든 콘텐츠를 검토할 의무가 있는 것은 아닙니다. 누군가 여러분의 권리를 침해하였다면, 고객센터를 통해 게시중단 요청에 대한 도움을 받으실 수 있습니다. 위와 관련된 구체적인 기준 및 이용제한 절차의 내용은 카카오 운영정책 에서 확인하실 수 있습니다. ⑤통합서비스에서는 회사가 보유하지 않은 일부 콘텐츠가 표시될 수 있습니다. 그러한 콘텐츠에 대해서는 콘텐츠를 제공한 주체가 단독으로 모든 책임을 부담하게 됩니다. 여러분이 통합서비스를 이용하더라도 다른 이용자의 콘텐츠에 대하여 어떠한 권리를 가지게 되는 것은 아닙니다. 여러분이 다른 이용자의 콘텐츠를 사용하기 위해서는 콘텐츠 소유자로부터 별도로 허락을 받아야 합니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":10,"title":"유료 서비스의 이용","text":"①통합서비스를 구성하는 개별 서비스의 대부분은 무료로 제공하고 있으나, 일부 개별 서비스는 유료로 제공될 수 있습니다. 예를 들면, 카카오톡에서 친구들과 무료로 메시지를 주고 받을 수 있으나 일부 이모티콘 등은 유료로 구매해야 친구들에게 보낼 수 있습니다. ②여러분이 회사가 제공하는 유료서비스를 이용하는 경우 이용대금을 납부한 후 이용하는 것을 원칙으로 합니다. 회사가 제공하는 유료서비스에 대한 이용요금의 결제 방법은 핸드폰결제, 신용카드결제, 일반전화결제, 계좌이체, 무통장입금, 선불전자지급수단 결제 등이 있으며 각 유료서비스마다 결제 방법의 차이가 있을 수 있습니다. 매월 정기적인 결제가 이루어지는 서비스의 경우 여러분 개인이 해당 서비스의 이용을 중단하고 정기 결제의 취소를 요청하지 않는 한 매월 결제가 이루어집니다. ③회사는 결제의 이행을 위하여 반드시 필요한 여러분의 개인정보를 추가적으로 요구할 수 있으며, 여러분은 회사가 요구하는 개인정보를 정확하게 제공하여야 합니다. ④본 조에서 정하지 않은 내용은 개별 서비스에 적용되는 유료서비스 약관(예: 카카오 유료서비스 이용약관 등)에서 정하며, 본 조의 내용과 개별 서비스에 적용되는 유료서비스 약관의 내용이 충돌하는 경우 개별 서비스에 적용되는 유료서비스 약관의 규정에 따릅니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":11,"title":"게시판 이용 상거래","text":"①여러분이 서비스를 이용하여 통신판매 또는 통신판매중개를 업으로 하는 경우 전자상거래 등에서의 소비자보호에 관한 법률(이하 ‘전자상거래법’)에 따른 의무사항을 준수하여야 합니다. ②여러분이 통신판매 또는 통신판매중개를 함에 있어 다른 이용자와 전자상거래 관련 분쟁이 발생하는 경우, 회사는 다른 이용자에게 소비자피해 구제 대행 신청을 할 수 있는 장치를 마련합니다. ③회사는 전자상거래법에 따라 신원정보를 입력하는 기능 등을 제공하여 여러분의 신원정보를 확인하고, 여러분과 다른 이용자 사이에 분쟁이 발생하여 전자상거래법에 따라 소비자피해 분쟁조정기구, 공정거래위원회, 시·도지사 또는 시장·군수·구청장이 신원정보 제공을 요구하는 경우 이에 협조합니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":12,"title":"통합서비스 이용 방법 및 주의점","text":"①여러분은 통합서비스를 자유롭게 이용할 수 있으나, 아래 각 호의 행위는 하여서는 안 됩니다. 1.이용 신청 또는 변경 시 허위 사실을 기재하거나, 다른 사람의 카카오계정 및 비밀번호를 도용, 부정하게 사용하거나, 다른 사람의 명의를 사용하거나 명의자의 허락 없이 문자메시지(SMS) 인증 등을 수행하는 행위 2.회사의 서비스 정보를 이용하여 얻은 정보를 회사의 사전 승낙 없이 복제 또는 유통시키거나 상업적으로 이용하는 행위 3.서비스 내에서 다운로드 또는 스트리밍을 통해 제공받은 음원을 사적 목적으로 이용하는 것 외에, 공공장소 및 영리를 목적으로 하는 영업장, 매장 등에서 재생하는 등의 방법으로 이용하는 행위 4.타인의 명예를 손상시키거나 불이익을 주는 행위 5.게시판 등에 음란물을 게재하거나 음란사이트를 연결(링크)하는 행위 6.회사 또는 제3자의 저작권 등 기타 권리를 침해하는 행위(국내외 제3자의 저작권 등을 침해하는 행위로서 회사가 IP 접속 차단 등 기술적인 조치를 통하여 타인에 대한 권리 침해 방지 조치를 취하였음에도 불구하고 이용자가 회사를 기망하는 수단과 방법 등을 통하여 서비스에 접속 하는 등 제3자의 저작권 등을 침해하는 행위를 포함합니다) 7.서비스 내에 회사나 제3자 등에 대한 허위의 사실을 게시하는 행위 8.공공질서 및 미풍양속에 위반되는 내용의 정보, 문장, 도형, 음성 등을 타인에게 유포하는 행위 9.통합서비스와 관련된 설비의 오동작이나 정보 등의 파괴 및 혼란을 유발시키는 컴퓨터 바이러스 감염 자료를 등록 또는 유포하는 행위 10.통합서비스의 운영을 방해하거나 안정적 운영을 방해할 수 있는 정보 및 수신자의 명시적인 수신거부의사에 반하여 또는 수신자의 명시적인 동의 없이 광고성 정보 또는 스팸메일(Spam Mail)을 전송하는 행위 11.회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위 12.타인으로 가장하는 행위 및 타인과의 관계를 허위로 명시하는 행위 13.다른 이용자의 개인정보를 수집, 저장, 공개하는 행위 14.자기 또는 타인에게 재산상의 이익을 주거나 타인에게 손해를 가하는 등 피해를 입힐 목적으로 허위의 정보를 유통시키는 행위 15.재물을 걸고 도박하거나 사행행위를 하는 행위 16.윤락행위를 알선하거나 음행을 매개하는 내용의 정보를 유통시키는 행위 17.수치심이나 혐오감 또는 공포심을 일으키는 말이나 음향, 글이나 화상 또는 영상을 계속하여 상대방에게 도달하게 하여 상대방의 일상적 생활을 방해하는 행위 18.관련 법령에 의하여 그 전송 또는 게시가 금지되는 정보(컴퓨터 프로그램 포함)의 전송 또는 게시 행위 19.회사 또는 관계사의 직원이나 운영자를 가장하거나 사칭하여 또는 타인의 명의를 도용하여 글을 게시하거나 E-mail, 카카오톡 메시지 등을 발송하는 행위 20.컴퓨터 소프트웨어, 하드웨어, 전기통신 장비의 정상적인 가동을 방해, 파괴할 가능성이 있는 소프트웨어 바이러스, 기타 다른 컴퓨터 코드, 파일, 프로그램을 포함하고 있는 자료를 게시하거나 E-mail, 카카오톡 메시지 등으로 발송하는 행위 21.스토킹(stalking), 허위 또는 악의적 신고 남용 등 다른 이용자를 괴롭히는 행위 22.1개월 이내 통합서비스 가입 및 유료서비스 구매 후 다시 해지하는 행위를 2회 이상 반복하는 등 통합서비스를 부당하게 악용하는 행위 23.기타 현행 법령, 본 약관 및 운영정책 등 회사가 제공하는 서비스 관련 세부지침을 위반하는 행위 ②여러분은 서비스의 이용 권한, 기타 이용계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할 수 없습니다. ③여러분의 자격 혹은 나이에 따라 아래 각 호처럼 통합서비스 이용의 일부가 제한될 수 있습니다. 1.만19세 미만의 이용자는(단, 만 19세에 도달하는 해의 1월 1일을 맞이한 자는 제외, 이하 본 조에서 동일함) 정보통신망법 및 청소년보호법의 규정에 의하여 청소년유해매체물은 이용할 수 없습니다. 2.청소년유해매체물을 이용하시기 위해서는 만 19세 이상이어야 하며, 정보통신망법 및 청소년보호법의 규정에 의하여 실명인증을 통해 본인 및 연령 인증을 받으셔야 합니다. 인증을 받지 않으시면, 해당 서비스의 이용이 제한됩니다. 3.만 19세 미만의 이용자의 서비스에 대하여 법정대리인의 요청 및 만19세 미만 이용자 본인의 동의가 있는 경우 개별 서비스의 전체 또는 일부의 이용이 일정기간 제한됩니다. ④회사는 음성대화 기능 등을 제공하는 일부 서비스 내에서 이용자간 신고가 있는 경우, 신고된 이용자의 음성정보를 저장 및 보관할 수 있으며 이 정보는 회사만 보유합니다. 회사는 이용자간 분쟁 조정, 민원 처리를 위한 목적에 한하여, 제 3 자는 법령에 따라 권한이 부여된 경우에 한하여 이 정보를 열람할 수 있습니다. 회사는 부정이용 방지 및 관리의 목적에 따라 신고 접수시부터 3년간 해당 정보를 3년간 보관 후 파기합니다. 단, 저 사양 단말기의 경우에는 신고된 음성정보가 저장되지 않을 수 있습니다. ⑤회사는 수사기관(경찰청 등)이 피싱 범죄에 이용중인 사실을 확인하여 법령 등에 따라 정당한 절차로 긴급차단을 요청하는 경우 여러분의 개별 서비스의 일부 또는 전부의 이용을 잠시 또는 계속하여 중단하는 이용 제한을 할 수 있습니다. 여러분이 이러한 이용 제한과 관련하여 이의가 있는 경우 이용정지를 요청한 수사기관에 이의제기를 할 수 있습니다. 수사기관에서 정당한 사유에 대한 소명이 확인된 경우 회사는 이용 제한을 해제할 수 있습니다. ⑥혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행위 등을 조사할 수 있고, 해당 게시물 등을 삭제 또는 임시 삭제하거나 여러분의 계정・통합서비스 전체 또는 통합서비스를 구성하는 일부 개별 서비스의 이용을 잠시 또는 계속하여 중단하거나, 통합서비스 재가입 또는 일부 개별 서비스의 재이용에 제한을 둘 수도 있습니다. 또한 여러분이 통합서비스와 관련된 설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 통합서비스 제공에 악영향을 미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다. ⑦이용 제한은 위반 활동의 누적 정도에 따라 한시적 제한에서 영구적 제한으로 단계적 제한하는 것을 원칙으로 하지만, 음란한 내용의 게시와 유포 및 사행성 도박 홍보 등 관련 법령에서 금지하는 명백한 불법행위나 타인의 권리침해로서 긴급한 위험 또는 피해 차단이 요구되는 사안에 대해서는 위반 활동 횟수의 누적 정도와 관계 없이 즉시 영구적으로 이용이 제한될 수 있습니다. ⑧본 조에서 정한 사항 및 그 밖에 통합서비스의 이용에 관한 자세한 사항은 카카오 운영정책 등을 참고해 주시기 바랍니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":13,"title":"이용계약 해지","text":"①여러분이 카카오계정 탈퇴를 하는 경우 통합서비스 이용계약도 자동으로 해지됩니다. ②통합서비스 이용계약 해지를 원하는 경우 여러분은 언제든지 서비스 내 제공되는 메뉴를 이용하여 해지 신청을 할 수 있으며,회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다. ③통합서비스 이용계약이 해지되면 관련 법령 및 카카오 개인정보 처리방침에 따라 여러분의 일정 정보를 보유하는 경우를 제외하고는 여러분의 정보나 여러분이 작성한 게시물 등 모든 데이터는 삭제됩니다. 다만, 여러분이 작성한 게시물이 제3자에 의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제3자의 게시물에 댓글 등 게시물을 추가하는 등의 경우에는 해당 게시물 및 댓글은 삭제되지 않으므로 반드시 해지 신청 전에 삭제하시기 바랍니다. ④전항에 따라 여러분이 삭제하지 않은 게시물은 다른 이용자의 정상적 서비스 이용을 위하여 필요한 범위 내에서 통합서비스 내에 삭제되지 않고 남아 있게 됩니다. ⑤유료서비스 이용계약의 해지는 여러분의 유료서비스 이용계약 해지 신청 및 회사의 승낙에 의해 성립하게 되고, 환불할 금액이 있는 경우 환불도 이루어 지게 됩니다. 다만 각 개별 서비스의 유료서비스에서 본 약관과 다른 계약해지 방법 및 효과를 규정하고 있는 경우 각 유료서비스 약관 및 관련 세부지침에서 정한 바에 따릅니다. ⑥통합서비스를 구성하는 일부 개별 서비스의 경우 일정기간 동안 해당 개별 서비스를 이용하지 않을 경우 여러분의 정보를 파기하거나 분리 보관할 수 있으며, 또는 해당 개별 서비스 기능의 일부 또는 전부를 이용할 수 없도록 제한할 수 있습니다. 자세한 사항은 개별 서비스의 세부지침에서 확인하실 수 있습니다. ⑦통합서비스 이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다. 다만, 여러분이 관련 법령, 본 약관 및 세부지침을 준수하지 않아 서비스의 이용이 중단된 상태에서 이용계약을 해지한 후 다시 이용계약 체결을 신청하는 경우에는 통합서비스 가입에 일정기간 시간적 제한이 있을 수 있습니다. 또한 통합서비스를 구성하는 일부 개별 서비스의 경우 다시 통합서비스 이용계약을 체결한 후에도 해당 개별 서비스를 바로 이용하는 것에는 제6조 제3항에서 정한 바와 같이 일정한 시간적 제한 등이 따를 수 있습니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":14,"title":"개인정보의 보호","text":"여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 통합서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 관련 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은 카카오 개인정보처리방침 등을 참고해 주시기 바랍니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":15,"title":"손해배상 등","text":"①회사는 관련 법령상 허용되는 한도 내에서 통합서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 여러분이 작성하는 등의 방법으로 통합서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다. ②회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 관련 법령에 따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징벌적 손해에 대한 책임을 부담하지 않습니다. 1.천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해 2.여러분의 귀책사유로 통합서비스 이용에 장애가 발생한 경우 3.통합서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해 4.제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해 5.제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해 6.제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해 7.전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에서 발생된 손해 8.기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해 ③회사는 회사의 고의 또는 과실이 없는 한 여러분이 통합서비스를 이용하여 기대하는 수익을 상실한 것에 대하여 책임을 지지 않으며 그 밖에 통합서비스를 통하여 얻은 자료로 인한 손해 등에 대하여도 책임을 지지 않습니다. ④회사는 회사의 과실이 없는 한 여러분 상호간 또는 여러분과 제3자 상호간에 통합서비스를 매개로 발생한 분쟁에 대해서는 개입할 의무가 없으며 이로 인한 손해를 배상할 책임도 없습니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":16,"title":"청소년보호","text":"통합서비스는 기본적으로 모든 연령대가 자유롭게 이용할 수 있는 공간으로서 유해 정보로부터 청소년을 보호하고 청소년의 안전한 인터넷 사용을 돕기 위해 정보통신망법에서 정한 청소년보호정책을 별도로 시행하고 있으며, 구체적인 내용은 통합서비스를 구성하는 개별 서비스 초기 화면 등에서 확인할 수 있습니다.","effective_date":"2026-05-29"},{"document":"카카오 통합서비스약관","article":17,"title":"통지 및 공지","text":"회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 카카오 고객센터 에 방문하여 의견을 개진할 수 있습니다. 서비스 이용자 전체에 대한 공지는 칠(7)일 이상 서비스 공지사항 란에 게시함으로써 효력이 발생합니다. 여러분에게 중대한 영향을 미치는 사항의 경우에는 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다.","effective_date":"2026-05-29"},{"document":"카카오 통합 약관","article":1,"title":"목적","text":"㈜카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 회사가 제공하는 다양한 인터넷과 모바일 서비스에 더 가깝게 다가갈 수 있도록 카카오 서비스 및 Daum 서비스(이하 통칭하여 ‘서비스’)에 통합 적용될 수 있는 카카오 통합 약관(이하 ‘본 약관’)을 마련하였습니다. 본 약관은 여러분이 서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다. • '카카오 서비스'라 함은 회사가 제공하는 1) “카카오” 브랜드를 사용하는 서비스(예:카카오톡) 또는 2) 카카오계정으로 이용하는 서비스(예: 브런치)를 의미하며, \"Daum\" 브랜드를 사용하는 서비스는 포함되지 않습니다. • ‘Daum 서비스’라 함은 회사가 제공하는 “Daum” 브랜드를 사용하는 서비스를 말합니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":2,"title":"약관의 명시, 효력 및 변경","text":"1. 본 약관의 내용은 회사가 제공하는 개별 서비스 또는 서비스 초기 화면에 게시하거나 기타의 방법으로 공지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다. 2. 회사는 필요한 경우 관련법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게 Daum 공지사항 또는 카카오 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게 여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정 또는 Daum 아이디 로 사용하는 이메일 주소로 이메일을 발송하거나, 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 보내거나, 서비스 내 전자쪽지 발송, 알림 메시지를 띄우는 등 합리적으로 가능한 방법으로 변경사항을 공지 또는 통지하겠습니다. 3. 회사가 전 항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 여러분이 개정약관에 동의하지 않을 경우 여러분은 제 14조 제1항에 따라 이용계약을 해지할 수 있습니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":3,"title":"약관 외 준칙","text":"본 약관에 규정되지 않은 사항에 대해서는 관련법령 또는 회사가 정한 서비스의 개별 이용약관, 운영정책 및 규칙 등(이하 ‘세부지침’)의 규정에 따릅니다. 또한 본 약관과 세부지침의 내용이 충돌할 경우 세부지침에 따릅니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":4,"title":"카카오계정 또는 Daum 아이디 생성 및 연결","text":"1. 카카오계정이란 여러분이 카카오 서비스를 사용하기 위하여 필요한 로그인 계정을 의미합니다. 카카오계정은 여러분이 약관에 동의하고 카카오계정 생성을 위해 필요한 일정 정보를 입력하시면, 카카오가 입력된 일정 정보를 인증한 후 가입을 승낙하는 절차로 생성됩니다. 2. Daum 아이디란 여러분이 Daum 서비스에서 본인을 식별하기 위해 미리 등록한 문자, 특수문자, 숫자 등의 조합으로, 여러분이 Daum 서비스약관 또는 본 약관에 동의하고 회원등록에 필요한 필수사항을 입력한 후 회원등록을 완료하면 회사가 승낙하는 절차로 생성됩니다. 다만, 여러분이 카카오계정으로 Dau m 서비스를 이용하는 경우 자동으로 추천된 Daum 아이디가 생성될 수 있습니다. 3. 카카오 서비스를 이용하기 위하여 반드시 카카오계정이 필요한 것은 아니나, 어떤 카카오 서비스는 카카오계정이 반드시 필요합니다. Daum 서비스약관에 동의한 Daum 아이디로는 Daum 서비스만 이용할 수 있습니다. 여러분이 Daum 서비스 중 카카오 서비스와 연결되는 기능을 이용하기 위해서는 카카오계정으로 Daum서비스를 이용하거나, 카카오계정과 Daum 아이디와의 연결이 필요합니다. 카카오계정으로 Daum 서비스를 이용하거나, 카카오계정에 기존에 등록한 Daum 아이디를 연결하면 카카오 서비스와 Daum 서비스에 설정한 정보, 서비스 이용기록 등을 카카오 서비스와 Daum 서비스에서 모두 이용할 수 있습니다. 회사는 서비스 회원 정책의 변경 등의 사유가 발생하였을 때, 회원에게 안내 후 계정 연결 서비스를 종료할 수 있습니다. 4. 여러분이 카카오계정으로 Daum 서비스를 이용하거나, 카카오계정과 Daum 아이디를 연결하면서 본 약관에 동의하면 그 이후부터는 본 약관만을 적용받게 되고, 카카오 서비스 약관 및 Daum 서비스 약관은 더 이상 적용되지 않습니다. 다만, 계정 연결 서비스를 종료할 경우에는 약관 변경에 따라 카카오 통합서비스 약관 또는 카카오 서비스 약관 및 Daum 서비스 약관이 적용될 수 있습니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":5,"title":"카카오계정 또는 Daum 아이디 생성 거절 및 유보","text":"1. 회사는 아래와 같은 경우에는 여러분의 카카오계정 및/또는 Daum 아이디의 생성을 승낙하지 않을 수 있습니다. 특히, 여러분이 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 카카오계정 및/또는 Daum 아이디를 생성할 수 있습니다. • 회사가 본 약관에 의해 여러분의 카카오계정 또는 Daum 아이디를 삭제하였던 경우 • 여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 카카오계정 또는 Daum 아이디를 생성하려 한 경우 • 카카오계정 또는 Daum 아이디 생성 시 필요한 정보를 입력하지 않거나 허위의 정보를 입력한 경우 • 기타 관련법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우 2. 만약, 여러분이 위 조건에 위반하여 카카오계정 및/또는 Daum 아이디를 생성한 것으로 판명된 때에는 회사는 즉시 여러분의 서비스 이용을 중단하거나 카카오계정 및 Daum 아이디를 삭제하는 등 적절한 제한을 할 수 있습니다. 3. 회사는 아래와 같은 경우에는 여러분의 카카오계정 및/또는 Daum 아이디 생성을 유보할 수 있습니다. • 제공 서비스 설비용량에 현실적인 여유가 없는 경우 • 서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우 • 기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":6,"title":"카카오계정 또는 Daum 아이디 등의 관리","text":"1. 카카오계정 및 Daum 아이디는 여러분 본인만 이용할 수 있으며, 다른 사람이 여러분의 카카오계정 및 D aum 아이디를 이용하도록 허락할 수 없습니다. 그리고 여러분은 다른 사람이 여러분의 카카오계정 및 D aum 아이디를 무단으로 사용할 수 없도록 직접 비밀번호를 관리하여야 합니다. 회사는 다른 사람이 여러분의 카카오계정 및/또는 Daum 아이디를 무단으로 사용하는 것을 막기 위하여 비밀번호 입력 및 추가적인 본인 확인 절차를 거치도록 할 수 있습니다. 만약 무단 사용이 발견된다면, 고객센터를 통하여 회사에게 알려주시기 바라며, 회사는 무단 사용을 막기 위한 방법을 여러분에게 안내하도록 하겠습니다. 2. 여러분은 서비스 내 설정 화면을 통하여 여러분의 정보를 열람하고 수정할 수 있습니다. 다만, 서비스의 제공 및 관리를 위해 필요한 카카오계정, Daum 아이디, 전화번호, 단말기 식별번호, 기타 본인확인정보 등 일부 정보는 수정이 불가능할 수 있으며, 수정하는 경우에는 추가적인 본인 확인 절차가 필요할 수 있습니다. 여러분이 서비스 이용 신청 시 알려주신 내용에 변동이 있을 때, 직접 서비스에서 수정하거나 이메일, 고객센터를 통하여 회사에 알려 주시기 바랍니다. 3. 여러분이 서비스 내 정보를 수정하지 않아 발생하는 손해에 대하여 회사는 책임을 부담하지 아니합니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":7,"title":"다양한 서비스 제공 및 변경 등","text":"1. 회사는 SNS, 게시판 서비스, 온라인 콘텐츠 제공 서비스, 위치기반서비스 등 여러분이 인터넷과 모바일로 즐길 수 있는 다양한 서비스를 제공합니다. 여러분은 스마트폰의 어플리케이션 스토어 등에서 서비스를 다운받아 설치하거나 직접 PC에 설치 혹은 웹페이지에 접속하여 서비스를 이용할 수 있습니다. 그런데 회사는 여러분이 원하는 다양한 서비스를 시시각각 제공하기 때문에 서비스의 자세한 내용은 별도로 알려드릴 수밖에 없습니다. 이러한 회사의 사정을 이해하여 주시길 바라며, 회사도 개별적인 서비스 이용방법을 어플리케이션 스토어와 각 서비스의 Q&A 센터, 해당 안내 및 고지사항에서 더 상세하게 안내하고 있으니 언제든지 확인하여 주시기 바랍니다. 2. 회사는 여러분이 서비스를 마음껏 이용할 수 있도록 이에 필요한 소프트웨어의 개인적이고 전 세계적이며 양도불가능하고 비독점적인 무상의 라이선스를 여러분에게 제공합니다. 단, 회사가 여러분에게 회사의 상표 및 로고를 사용할 권리를 부여하는 것은 아니라는 점은 잊지 말아주시기 바랍니다. 3. 회사는 더 나은 서비스를 위하여 서비스에 필요한 소프트웨어의 업데이트 버전을 제공할 수 있습니다. 소프트웨어의 업데이트에는 중요한 기능의 추가 또는 불필요한 기능의 제거 등이 포함되어 있습니다. 여러분들도 서비스를 즐겁게 이용할 수 있도록 꾸준히 업데이트를 하여 주시기 바랍니다. 4. 회사는 더 나은 서비스의 제공을 위하여 여러분에게 서비스의 이용과 관련된 각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 서비스에 표시하거나 여러분의 메일 계정으로 직접 발송할 수 있습니다. 5. 서비스 이용 중 시스템 오류 등 문제점을 발견하신다면 언제든지 카카오 고객 센터로 알려주시기 바랍니다. 6. 여러분이 서비스를 이용하는 과정에서 Wi-Fi 무선인터넷을 사용하지 않고, 가입하신 이동통신사의 무선인터넷에 연결하여 이용하는 경우 이동통신사로부터 여러분에게 별도의 데이터 통신요금이 부과되는 점을 유의하여 주시기 바랍니다. 서비스 이용 과정에서 발생하는 데이터 통신요금은 여러분이 여러분의 비용과 책임 하에 이동통신사에 납부하셔야 합니다. 데이터 통신요금에 대한 자세한 안내는 여러분이 가입하신 이동통신사에 문의하시기 바랍니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":8,"title":"서비스 이용 방법 및 주의점","text":"1. 여러분은 서비스를 자유롭게 이용할 수 있으나, 아래와 같이 서비스를 잘못된 방법으로 이용할 수 없다는 점을 잊지 말아주셨으면 합니다. • 여러분은 잘못된 방법으로 서비스의 제공을 방해하거나 회사가 안내하는 방법 이외의 다른 방법을 사용 하여 서비스에 접근할 수 없습니다. • 다른 서비스 이용자의 정보를 무단으로 수집, 이용하거나 다른 사람들에게 제공하는 행위도, 수신자의 명시적 수신거부 의사에 반하여 또는 수신자의 명시적인 동의 없이 광고성 정보를 전송하거나 서비스를 영리 목적으로 이용하는 것도, 음란 정보나 저작권 침해, 회사나 제3자 등에 대한 허위의 사실을 게시하는 정보 등 공서양속 및 법령에 위반되는 내용의 정보 등을 발송하거나 게시하는 행위도 금지됩니다. • 회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위도 금지됩니다. 2. 여러분은 서비스의 이용권한, 기타 이용 계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할 수 없습니다. 3. 카카오는 음성대화 기능 등을 제공하는 일부 서비스 내에서 이용자간 신고가 있는 경우, 신고된 이용자의 음성정보를 저장 및 보관할 수 있으며 이 정보는 회사만 보유합니다. 카카오는 이용자간 분쟁 조정, 민원 처리를 위한 목적에 한하여, 제 3 자는 법령에 따라 권한이 부여된 경우에 한하여 이 정보를 열람할 수 있습니다. 카카오는 부정이용 방지 및 관리의 목적에 따라 신고 접수시부터 3년간 해당 정보를 3년간 보관 후 파기합니다. 단, 저 사양 단말기의 경우에는 신고된 음성정보가 저장되지 않을 수 있습니다. 4. 혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행위 등을 조사할 수 있고, 해당 게시물 등을 삭제 또는 임시 삭제하거나 여러분의 서비스 이용을 잠시 또는 계속하여 중단하거나, 재가입에 제한을 둘 수도 있습니다. 또한 여러분이 서비스와 관련된 설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 서비스 제공에 악영향을 미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다. 5. 회사는 법령에서 정하는 기간 동안 여러분이 서비스를 이용하기 위해 로그인 혹은 접속한 기록이 없는경 우 여러분이 등록한 이메일주소, 휴대폰번호로 이메일, 문자메시지 또는 카카오톡 메시지를 보내는 등 기타 유효한 수단으로 통지 후 여러분의 정보를 파기하거나 분리 보관할 수 있으며, 이로 인해 서비스 이용을 위한 필수적인 정보가 부족할 경우 이용계약이 해지될 수도 있습니다. 6. 회사는 스팸성 메일(피싱, 바이러스 유포, 개인정보 탈취 등 각종 불법 및 사행성 스팸을 의미합니다)로부터 이용자를 보호하기 위해 수발신 메일에 대한 스팸 대응 및 보안 조치를 합니다. 더불어 유관기관의 권고가 있거나 이용자 보호를 위하여 필요하다고 판단하는 경우 이에 따른 추가적인 스팸 운영정책 및 기능을 제공합니다. 7. 본 조에서 정한 사항 및 그 밖에 서비스의 이용에 관한 자세한 사항은 서비스 운영정책 등 을 참고해 주시기 바랍니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":9,"title":"게시물의 관리","text":"1. 여러분의 게시물이 정보통신망 이용촉진 및 정보보호 등에 관한 법률(이하 ‘정보통신망법’)및 저작권법등 관련법에 위반되는 내용을 포함하는 경우, 권리자는 회사에 관련법이 정한 절차에 따라 해당 게시물의 게시중단 및 삭제 등을 요청할 수 있으며, 회사는 관련법에 따라 조치를 취합니다. 2. 회사는 권리자의 요청이 없는 경우라도 권리침해가 인정될 만한 사유가 있거나 기타 회사의 정책 및 관련 법에 위반되는 경우에는 관련법에 따라 해당 게시물에 대해 임시조치 등을 취할 수 있습니다. 3. 위와 관련된 세부절차는 정보통신망법 및 저작권법이 규정한 범위 내에서 회사가 정한 ‘권리침해 신고(카카오 서비스, Daum 서비스)’절차에 따릅니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":10,"title":"권리의 귀속 및 저작물의 이용","text":"1. 여러분은 사진, 글, 정보, (동)영상, 카카오 서비스, Daum 서비스 또는 회사에 대한 의견이나 제안 등 콘텐츠(이하 ‘게시물’)를 서비스에 게시할 수 있으며, 이러한 게시물에 대한 저작권을 포함한 지적재산권은 당연히 권리자가 계속하여 보유합니다. 2. 여러분은 카카오 서비스 또는 Daum 서비스에 게시한 게시물에 대한 사용, 저장, 수정, 복제, 공중송신, 전시, 배포 등의 방식으로 이용할 수 있도록 사용을 허락하는 전 세계적이고 영구적인 라이선스를 회사에게 제공하게 됩니다. 본 라이선스에서 여러분이 회사에게 부여하는 권리는 서비스를 운영, 개선, 홍보하고 새로운 서비스를 개발하기 위한 범위 내에서 사용됩니다. 이러한 목적 범위 내에서 회사와 명시적인 업무계약을 체결한 상대방 또는 다른 이용자에 대한 서브라이선스 또한 여기에 포함됩니다. 또한, 서비스의 개선 및 연구개발 목적으로 회사 및 회사의 계열사에서 게시물을 사용할 수 있습니다. 본 라이선스는 여러분이 서비스의 사용을 중단하거나 카카오계정 및/또는 Daum 아이디를 탈퇴한 후에도 존속하게 됩니다. 일부 서비스에서는 여러분이 제공한 콘텐츠에 접근하거나 이를 삭제하는 방법을 제공할 수 있습니다( 다만 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도 있습니다). 또한 일부 서비스에서는 제공된 콘텐츠에 대한 회사의 사용 범위를 제한하는 설정이 있습니다. 3. 여러분은 회사에 제공한 콘텐츠에 대해 회사에 라이선스를 부여하기 위해 필요한 권리를 보유해야 합니다 . 이러한 권리를 보유하지 않아 발생하는 모든 문제에 대해서는 게시자가 책임을 부담하게 됩니다. 또한, 여러분은 음란하거나 폭력적이거나 기타 공서양속 및 법령에 위반하는 콘텐츠를 공개 또는 게시할 수 없습니다. 4. 회사는 여러분의 콘텐츠가 법령에 위반되거나 음란 또는 청소년에게 유해한 게시물, 차별 갈등을 조장하는 게시물, 도배·광고·홍보·스팸성 게시물, 계정을 양도 또는 거래하는 내용의 게시물, 타인을 사칭하는 게시물 등 이라고 판단되는 경우 이를 삭제하거나 게시를 거부할 수 있습니다. 다만 회사가 모든 콘텐츠를 검토할 의무가 있는 것은 아닙니다. 누군가 여러분의 권리를 침해하였다면, 고객센터를 통해 게시중단요 청에 대한 도움을 받으실 수 있습니다. 위와 관련된 구체적인 기준 및 이용제한 절차의 내용은 카카오 운 영정책에서 확인하실 수 있습니다. 5. 서비스에서는 회사가 보유하지 않은 일부 콘텐츠가 표시될 수 있습니다. 그러한 콘텐츠에 대해서는 콘텐츠를 제공한 주체가 단독으로 모든 책임을 부담하게 됩니다. 여러분이 서비스를 이용하더라도 다른 이용자의 콘텐츠에 대하여 어떠한 권리를 가지게 되는 것은 아닙니다. 여러분이 다른 이용자의 콘텐츠를 사용하기 위해서는 콘텐츠 소유자로부터 별도로 허락을 받아야 합니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":11,"title":"유료 서비스의 이용","text":"1. 회사는 무료로 서비스를 제공하고 있으나, 일부 서비스의 경우 유료로 제공할 수 있습니다. 예를 들면, 카카오톡에서 친구들과 무료로 메시지를 주고 받을 수 있으나 일부 이모티콘 등은 유료로 구매해야 친구들에게 보낼 수 있으며, Daum 메일은 무료로 이용할 수 있으나 프리미엄 메일은 유료로 이용할 수 있습니다. 2. 여러분이 회사가 제공하는 유료서비스를 이용하는 경우 이용대금을 납부한 후 이용하는 것을 원칙으로 합니다. 회사가 제공하는 유료서비스에 대한 이용요금의 결제 방법은 핸드폰결제, 신용카드결제, 일반전화결제, 계좌이체, 무통장입금, 선불전자지급수단 결제 등이 있으며 각 유료서비스마다 결제 방법의 차이가 있을 수 있습니다. 매월 정기적인 결제가 이루어지는 서비스의 경우 여러분 개인이 해당 서비스의 이용을 중단하고 정기 결제의 취소를 요청하지 않는 한 매월 결제가 이루어집니다. 3. 회사는 결제의 이행을 위하여 반드시 필요한 여러분의 개인정보를 추가적으로 요구할 수 있으며, 여러분은 회사가 요구하는 개인정보를 정확하게 제공하여야 합니다. 4. 여러분 개인의 귀책사유로 이용요금을 환불하는 경우 일반적인 방법은 아래와 같습니다. • 회사가 제공하는 유료서비스가 결제 후 1회의 이용만으로 서비스의 이용이나 구매가 완료되는 서비스인 경우 해당 서비스를 이용한 후에는 환불이 불가능합니다. 단, 1회의 구매 완료 후 그 사용기한이 무제한 인 아바타, 배경음악, 스킨 등의 서비스는 구매 완료일로부터 1년 이내에만 환불이 가능하며 환불금액은 구입금액*(365-사용일수/365)로 합니다. • 회사가 제공하는 유료서비스가 결제 후 1개월(결제 기준) 이하로 지속되는 서비스인 경우 해지일로부터 이용일수에 해당하는 금액을 제외한 나머지 금액을 환불합니다. 본 항의 규정은 일(1)개월 단위로 매월 결제되는 서비스의 경우에도 적용됩니다. • 회사가 제공하는 유료서비스가 결제 후 1개월(결제 기준)을 초과하여 지속되는 서비스인 경우 해지일로 부터 이용일수에 해당하는 금액과 총 남은 이용일수의 10%를 제외한 금액을 환불합니다. 단, 유료 서비스 이용 개시일로부터 7일 이내에 해지를 요구하는 경우 이용일수에 해당하는 금액만을 제외하고 환 불합니다. 5. 상기의 규정에도 불구하고 아래 각 호의 경우에는 여러분 개인이 결제한 전액을 환불합니다. 단, 1회의 구매 완료 후 그 사용기한이 무제한인 아바타, 배경음악, 스킨 등의 서비스는 구매 완료일로부터 1년 이내 일 경우에만 환불합니다. • 여러분이 결제를 완료한 후 서비스를 이용한 내역이 없는 경우 • 서비스 장애 또는 회사가 제시한 최소한의 기술사양을 충족하였음에도 불구하고 회사의 귀책사유로 서비스를 이용하지 못한 경우 • 여러분이 구매한 서비스가 제공되지 않은 경우 • 제공되는 서비스가 표시·광고 등과 상이하거나 현저한 차이가 있는 경우 • 제공되는 서비스의 결함으로 서비스의 정상적인 이용이 현저히 불가능한 경우 6. 여러분은 이용요금에 대하여 이의를 제기할 수 있습니다. 단, 이용요금에 관한 이의는 그 사유 발생을 안 날로부터 1월, 그 사유가 발생한 날로부터 3월 이내에 제기하여야 합니다. 7. 회사는 과오금이 발생한 경우 또는 전액 환불의 경우 이용대금의 결제와 동일한 방법으로 환불하여야 합니다. 다만, 동일한 방법으로 환불이 불가능하거나 서비스의 중도해지로 인한 부분 환불 등의 경우에는 회사가 정하는 별도의 방법으로 환불합니다. 회사는 환불 의무가 발생한 날로부터 3영업일 이내에 환불을 진행하며, 환불이 지연되는 경우 지연이자율은 연리 11%로 합니다. 단, 환불에 여러분의 협조가 필요한 경우에 여러분의 귀책사유로 인한 환불 지연에 대해서는 지연이자를 지급하지 않습니다. 환불에 소요되는 비용은 여러분의 귀책사유로 인한 환불의 경우에는 여러분이, 회사의 귀책사유로 인한 환불의 경우에는 회사가 각각 부담합니다. 8. 본 약관의 유료서비스 규정과 각 각 개별 유료서비스 약관의 내용이 충돌하는 경우 각 개별약관의 규정에 따릅니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":12,"title":"게시판 이용 상거래","text":"1. 여러분이 서비스를 이용하여 통신판매 또는 통신판매중개를 업으로 하는 경우 전자상거래 등에서의 소비자보호에 관한 법률(이하 ‘전자상거래법’)에 따른 의무사항을 준수하여야 합니다. 2. 여러분이 통신판매 또는 통신판매중개를 함에 있어 다른 이용자와 전자상거래 관련 분쟁이 발생하는 경우 , 회사는 다른 이용자에게 소비자피해 구제 대행 신청을 할 수 있는 장치를 마련합니다. 3. 회사는 전자상거래법에 따라 신원정보를 입력하는 기능 등을 제공하여 여러분의 신원정보를 확인하고, 여러분과 다른 이용자 사이에 분쟁이 발생하여 전자상거래법에 따라 소비자피해 분쟁조정기구, 공정거래 위 원회, 시도지사 또는 시장 군수 구청장이 신원정보 제공을 요구하는 경우 이에 협조합니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":13,"title":"서비스의 이용, 변경 및 종료","text":"1. 회사는 서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 장비의 유지· 보수를 위한 정기 또는 임시 점검 또는 다른 상당한 이유로 서비스의 제공이 일시 중단될 수 있으며, 이 때에는 미리 서비스 제공화면에 공지하겠습니다. 만약, 회사로서도 예측할 수 없는 이유로 서비스가 중단된 때에는 회사가 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하고, 2시간 이상 복구가 지연되는 경우 Daum 공지사항 또는 카카오 서비스 공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다. 2. 회사의 서비스 제공을 위해 계약한 CP와의 계약 종료 및 변경, 서비스/회원 정책의 변경, 신규서비스 개 시 등의 사유로 서비스의 내용이 변경되거나, 서비스가 종료될 수도 있습니다. 서비스 변경 사항 또는 종료는 개별 서비스의 화면 또는 공지사항 란에 게시하여 여러분들께 알려드리겠습니다. 여러분께 중대한 영향을 미치는 서비스 변경 사항이나 종료는 전자메일(전자메일이 없는 경우 서비스 내 알림 등 별도의 전자적 수단) 또는 전화번호로 문자메세지를 발송하는 방법 등으로 개별적으로 알려드리겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":14,"title":"이용계약 해지","text":"1. 여러분이 서비스의 이용을 더 이상 원치 않는 때에는 언제든지 서비스 내 제공되는 메뉴를 이용하여 서비스 이용계약의 해지 신청을 할 수 있으며, 회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다. 2. 이용계약이 해지되면 법령 및 개인정보 처리방침에 따라 여러분의 정보를 보유하는 경우를 제외하고는 여러분의 정보나 여러분이 작성한 게시물 등 모든 데이터는 삭제됩니다. 다만, 여러분이 작성한 게시물이 제3자에 의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제3자의 게시물에 댓글 등 게시물을 추가하는 등의 경우에는 해당 게시물 및 댓글이 삭제되지 않으므로 반드시 해지신청 전에 삭제하신 후 탈퇴하시기 바랍니다. 3. 또한, 여러분은 다양한 서비스 중에서 일부 서비스만을 선택적으로 해지하실 수 있으며, 이 경우에는 해지된 서비스에 대한 데이터만 삭제되며, 다른 서비스 이용을 위한 카카오계정 및 Daum 아이디는 삭제되지 않고 남아 있게 됩니다. 4. 유료서비스 이용계약의 해지는 여러분의 서비스 해지 신청 및 회사의 승낙에 의해 성립하게 되고, 환불할 금액이 있는 경우 환불도 이루어 지게 됩니다. 다만 각 개별 유료서비스에서 본 약관과 다른 계약해지 방법 및 효과를 규정하고 있는 경우 각 개별약관의 규정에 따릅니다. 5. 이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다. 다만, 일부 서비스의 경우 다시 이용계약을 체결함에 있어 시간적 제한 등이 따를 수 있으며 이에 대한 구체적인 내용은 세부지침에서 확인하실 수 있습니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":15,"title":"개인정보의 보호","text":"여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은 Daum 개인정보처리방침과 카카오 개인정보처리방침을 참고하여 주십시오.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":16,"title":"위치기반서비스 제공","text":"1. 회사는 여러분의 실생활에 더욱 보탬이 되는 유용한 서비스를 제공하기 위하여 서비스에 위치기반서비스를 포함시킬 수 있습니다. 2. 회사의 위치기반서비스는 여러분의 단말기기의 위치정보를 수집하는 위치정보사업자로부터 위치정보를 전 달받아 제공하는 무료서비스이며, 구체적으로는 아래와 같습니다. • 여러분의 현재 위치 또는 특정 위치를 다른 이용자와 공유하거나 그와 관련된 게시물을 작성할 수 있도록 하는 서비스(장소공유서비스) • 여러분의 현재 위치를 이용한 생활 정보나 광고성 정보를 제공하는 서비스(정보제공서비스) • 여러분이 보유하는 사진 등 콘텐츠에 기록되거나 콘텐츠와 결합된 위치정보를 활용하여 다른 이용자와 콘텐츠를 공유하도록 도와주는 서비스(콘텐츠공유서비스) 3. 여러분이 14세 미만 이용자로서 개인위치정보를 활용한 위치기반서비스를 이용하기 위해서는 회사는 여러분의 개인위치정보를 이용 또는 제공하게 되며, 이 경우 부모님 등 법정대리인의 동의가 먼저 있어야 합니다. 만약 법정대리인의 동의 없이 위치기반서비스가 이용된 것으로 판명된 때에는 회사는 즉시 여러분의 위치기반서비스 이용을 중단하는 등 적절한 제한을 할 수 있습니다. 4. 여러분(14세 미만 이용자의 법정대리인 포함)은 서비스와 관련된 개인위치정보의 이용, 제공 목적, 제공받는 자의 범위 및 위치기반서비스의 일부에 대하여 동의를 유보하거나, 이용·제공에 대한 동의의 전부 또는 일부 철회할 수 있으며, 일시적인 중지를 요구할 수 있습니다. 회사는 위치정보의 보호 및 이용 등에 관한 법률의 규정에 따라 개인위치정보 및 위치정보 이용·제공사실 확인자료를 6개월 이상 보관하며, 여러분이 동의의 전부 또는 일부를 철회한 때에는 회사는 철회한 부분에 해당하는 개인위치정보 및 위치정보 이용·제공사실 확인자료를 지체 없이 파기하겠습니다. 5. 여러분(14세 미만 이용자의 법정대리인 포함)은 회사에 대하여 여러분에 대한 위치정보 이용·제공사실 확 인자료나, 여러분의 개인위치정보가 법령에 의하여 제3자에게 제공되었을 때에는 그 이유 및 내용의 열람 또는 고지를 요구할 수 있고, 오류가 있는 때에는 정정을 요구할 수 있습니다. 만약, 회사가 여러분의 개인위치정보를 여러분이 지정하는 제3자에게 직접 제공하는 때에는 법령에 따라 개인위치정보를 수집한 스 마트폰 등으로 여러분에게 개인위치정보를 제공받는 자, 제공 일시 및 제공 목적을 즉시 통보하겠습니다. 6. 회사는 8세 이하의 아동 등(금치산자, 중증 정신장애인 포함)의 보호의무자가 개인위치정보의 이용 또는 제공에 서면으로 동의하는 경우에는 해당 본인의 동의가 있는 것으로 보며, 이 경우 보호의무자는 개인위치정보주체의 권리를 모두 행사할 수 있습니다. 7. 만약 회사가 제공하는 위치기반서비스와 관련하여 여러분의 권리를 침해당했거나 권리행사가 필요한 경우 고객센터를 통해 도움을 받으실 수 있으며, 여러분과 회사 간의 위치정보와 관련한 분쟁에 대하여 협의가 어려운 때에는 여러분은 위치정보의 보호 및 이용 등에 관한 법률 제 28조 제2항 및 개인정보보호법 제43조의 규정에 따라 개인정보 분쟁조정위원회에 조정을 신청할 수 있습니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":17,"title":"인증서비스","text":"1. 본 조에서 사용하는 용어의 정의는 다음과 같습니다. • 인증서비스 : 회사가 제공하는 전자서명과 인증서를 활용한 일체의 서비스를 말합니다. • 전자서명: 서명자의 신원을 확인하고 서명자가 해당 전자문서에 서명하였다는 사실을 나타내는데 이용하기 위하여 전자문서에 첨부되거나 논리적으로 결합된 전자적 형태의 정보를 말합니다. • 인증서: 인증서라 함은 회사가 인증서비스를 통하여 발급하는 전자서명생성정보가 회원에게 유일하게 속 한다는 사실 등을 확인하고 이를 증명하는 전자적 정보를 말합니다. • 전자서명생성정보: 전자서명을 생성하기 위하여 이용하는 전자적 정보를 말합니다. • 이용기관: 인증회원의 전자서명 및 인증서를 바탕으로 한 거래 등을 위하여 인증서비스를 이용하려는 제3자를 말합니다. • 인증회원 : 회사로부터 전자서명생성정보를 인증 받은 회원을 말합니다. 2. 회사는 전자서명생성정보 및 인증서를 발급하고, 전자서명과 인증서를 활용한 각종 서비스를 아래 각 호 와 같이 인증회원에게 제공합니다. 이 때 회사는 필요한 경우 인증서비스의 유형 및 종류를 추가하거나, 부가 서비스를 별도로 제공할 수 있습니다. • 전자서명생성정보 및 인증서 발급 • 전자서명 및 인증서를 활용한 각종 서비스 • 이용기관 로그인 및 신원확인을 위한 간편인증 • 기타 전자서명인증업무 운영준칙에서 정하는 서비스 3. 회원은 회사가 정하는 방법에 따라 정확한 정보만을 제공하여 인증서비스에 가입하여야 하며, 인증서를 발급받음과 동시에 인증회원으로 전환됩니다. 인증서는 명의자 당 1개의 카카오계정에서 1대의 기기에만 발급 됩니다. 만일 인증회원이 다른 기기 또는 다른 카카오계정에서 인증서를 재발급하는 경우 기존에 발급받은 인 증서는 자동 폐지됩니다. 4. 인증회원은 회사가 정한 방법에 따라 인증서비스를 이용하여야 합니다. 또한 인증회원은 자신의 전자서명 생성정보와 인증서 및 이와 관련된 모든 정보를 안전하게 관리하고 인증서비스 이용 기간 중 회사에 제공한 정보 및 인증서에 포함된 정보가 정확하고 완전하게 유지되도록 하여야 합니다. 인증회원은 자신의 전자서명 생성정보와 인증서 및 이에 관련된 정보를 타인에게 양도, 증여, 판매, 사용 허락할 수 없으며, 분실, 훼손, 도난 또는 유출되거나 그러할 위험이 있다고 인지한 경우 즉시 그 사실을 회사에 통지하여야 합니다. 5. 회사는 다음 각 호의 경우 인증서의 신청 및 발급을 제한하거나 발급된 인증서를 인증회원의 동의 없이 폐지할 수 있습니다. • 피성년후견인 또는 피한정후견인이 법정대리인의 동의 없이 가입한 경우 • 타인 명의의 신청 및 정보 도용 등 신청 내용이 허위의 사실이라 판단되는 경우 • 회사가 제시하는 인증 절차를 완료하지 못하거나, 회사가 정하지 않은 비정상적인 방법으로 시스템에 접 근하여 인증서비스에 가입하는 경우 • 회사로부터 이용 정지를 당하거나, 법령 또는 본 약관을 위반하는 등의 이유로 서비스 이용 계약이 해지된 회원이 재이용신청을 하는 경우 • 기타 회원의 귀책사유로 발급이 곤란한 경우 또는 회사가 정한 이용신청 요건이 충족되지 않은 경우 6. 인증회원은 인증서비스를 자유롭게 이용할 수 있으나, 아래 각 호의 행위는 하여서는 안 됩니다. • 회사가 정하지 않은 비정상적인 방법으로 시스템에 접근하거나 인증서비스를 이용하는 행위 • 부정한 방법으로 인증서를 발급받거나 행사하는 등 인증서비스를 불법적 또는 부당한 용도로 사용하는 행위 7. 회사는 다음 각 호의 경우 인증회원에게 발급한 인증서 이용의 일부 또는 전부를 제한할 수 있으며, 인증 회원의 동의 없이 인증서를 즉시 폐지할 수 있습니다. • 인증서의 유효기간이 경과한 경우 • 인증회원이 인증서의 비밀번호를 연속하여 제한 횟수 이상 잘못 입력한 경우 • 인증회원의 카카오계정에 등록된 카카오톡 전화번호가 변경된 경우 • 인증회원의 사망, 구속 등으로 신원확인이나 전자거래가 불가능한 경우 • 인증서비스 가입 시 본인확인기관에서 전달받은 연계정보(CI)가 국적, 성별 등의 변경으로 더이상 유효하지 않음이 확인된 경우 • 회사가 인증서비스와 관련된 보안절차나 인증회원의 전자서명생성정보 유출과 같은 보안상의 이유로 기 발급된 인증서의 이용제한이 필요한 경우 • 전시, 사변, 천재지변 또는 이에 준하는 비상사태가 발생하거나 발생할 우려가 있는 경우 • 회사 고객센터 등을 통해서 인증서의 분실신고가 접수된 경우 • 인증회원의 인증서가 부정하게 사용된 사실을 회사가 인지한 경우 등 인증회원이 본 약관 및 운영정책을 포함한 회사의 서비스 이용 정책을 위반하거나 위반할 우려가 있다고 회사가 판단하는 경우 • 기타 인증서비스의 안전성과 신뢰성을 저해할 우려가 있는 경우 8. 회사는 인증서를 사용하는 인증회원과 이용기관 상호간 거래에 대하여 어떠한 책임도 부담하지 않으며, 회사는 인증회원과 이용기관의 귀책사유로 인하여 발생한 손해에 대하여 회사의 귀책사유가 없는 경우 책임을 부담하지 않습니다. 9. 본 조에서 정하고 있는 내용 외에 인증서비스와 관련된 상세한 사항은 전자서명법 등을 포함한 관련법령 및 회사가 별도로 정한 전자서명인증업무준칙에 따르며, 회사는 인증서비스 공지사항 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":18,"title":"손해배상 등","text":"1. 회사는 법령상 허용되는 한도 내에서 서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 회원이 작성하는 등의 방법으로 서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다. 2. 회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 법령에 따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징벌적 손해에 대한 책임을 부담하지 않습니다. • 천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해 • 여러분의 귀책사유로 서비스 이용에 장애가 발생한 경우 • 서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해 • 제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해 • 제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해 • 제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해 • 전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에서 발생된 손해 • 기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":19,"title":"청소년보호","text":"모든 연령대가 자유롭게 이용할 수 있는 공간으로써 유해 정보로부터 청소년을 보호하고 청소년의 안전한 인터넷 사용을 돕기 위해 정보통신망법에서 정한 청소년보호정책을 별도로 시행하고 있으며, 구체적인 내용은 서비스 초기 화면 등에서 확인할 수 있습니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":20,"title":"통지 및 공지","text":"회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 고객센터에 방문하여 의견을 개진할 수 있습니다. 회사는 카카오계정 또는 Daum 아이디로 사용하는 이메일 주소로 이메일을 발송하거나, 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 보내거나, 서비스 내 전자쪽지 발송, 알림 메시지를 띄우는 등 합리적으로 가능한 방법으로 여러분에게 공지 또는 통지하며, 서비스 이용자 전체에 대한 공지는 칠(7)일 이상 서비스 공지사항 란에 게시함으로써 효력이 발생합니다.","effective_date":"2022-08-25"},{"document":"카카오 통합 약관","article":21,"title":"분쟁의 해결","text":"본 약관 또는 서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 서비스 이용과 관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사소송법의 관할법원에 소를 제기할 수 있습니다. • 공고일자 : 2022년 8월 18일 • 시행일자 : 2022년 8월 25일 서비스(위치기반서비스 포함) 관련 문의사항이 있으시면 언제든지 고객센터에 방문 또는 연락해 주시기 바랍니다.","effective_date":"2022-08-25"}]''')

_counts = Counter(a["document"] for a in ARTICLES)
_EXPECTED = {"카카오계정 약관": 17, "카카오 통합서비스약관": 18,
             "카카오 통합 약관": 21, "카카오 위치정보 이용약관": 16}
assert set(_counts) == set(OFFICIAL_DOCUMENT_NAMES), f"문서명 불일치: {sorted(_counts)}"
for _doc, _n in _EXPECTED.items():
    assert _counts[_doc] == _n, f"{_doc}: 조 {_n}개여야 하는데 {_counts[_doc]}개"
for _doc in OFFICIAL_DOCUMENT_NAMES:
    _nums = sorted(a["article"] for a in ARTICLES if a["document"] == _doc)
    assert _nums == list(range(1, len(_nums) + 1)), f"{_doc}: 조번호가 끊김 {_nums}"
print(f"[인덱스] 조항 {len(ARTICLES)}개 — "
      + " / ".join(f"{d} {_counts[d]}" for d in OFFICIAL_DOCUMENT_NAMES))


# ----- 1-2. 검색 ---------------------------------------------------------------------
import math
import re
import unicodedata
from collections import Counter

_WORD_RE = re.compile(r"[0-9A-Za-z가-힣]+")
_HANGUL_RE = re.compile(r"[가-힣]+")

# 긴 조사부터 검사해야 '에게서'가 '에'로 잘못 잘리지 않는다.
_JOSA = (
    "으로써", "에게서", "이라고", "에서는", "으로는", "에서", "에게", "으로", "라고",
    "까지", "부터", "보다", "처럼", "이나", "마다", "조차", "한테", "와의", "과의",
    "의", "가", "이", "은", "는", "을", "를", "와", "과", "로", "도", "만", "에",
)


def tokenize(text, josa=True, bigram=True):
    tokens = []
    for word in _WORD_RE.findall(unicodedata.normalize("NFC", str(text))):
        tokens.append(word)
        if not _HANGUL_RE.fullmatch(word):
            continue
        if josa and len(word) > 2:
            for particle in _JOSA:
                if word.endswith(particle) and len(word) - len(particle) >= 2:
                    tokens.append(word[: -len(particle)])
                    break
        if bigram and len(word) >= 2:
            tokens.extend(word[i:i + 2] for i in range(len(word) - 1))
    return tokens


class BM25:
    def __init__(self, corpus_tokens, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.n_docs = len(corpus_tokens)
        self.lengths = [len(doc) for doc in corpus_tokens]
        self.avgdl = sum(self.lengths) / self.n_docs
        self.term_freqs = [Counter(doc) for doc in corpus_tokens]
        df = Counter()
        for doc in corpus_tokens:
            df.update(set(doc))
        self.idf = {
            term: math.log(1 + (self.n_docs - n + 0.5) / (n + 0.5)) for term, n in df.items()
        }

    def scores(self, query_tokens):
        out = [0.0] * self.n_docs
        for i, tf in enumerate(self.term_freqs):
            dl, total = self.lengths[i], 0.0
            for term in query_tokens:
                freq = tf.get(term)
                if not freq:
                    continue
                total += self.idf[term] * freq * (self.k1 + 1) / (
                    freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                )
            out[i] = total
        return out


class Index:
    """조항 목록을 받아 BM25 로 검색한다."""

    # 기본값은 공개 10 + 검증 18문항으로 측정한 최적 조합이다. 조사 제거는
    # 검증 MRR 을 0.8380 -> 0.8333 으로 떨어뜨려 끈다.
    def __init__(self, articles, josa=False, bigram=True):
        self.articles = articles
        self.opts = {"josa": josa, "bigram": bigram}
        self.bm25 = BM25([tokenize(self._text(a), **self.opts) for a in articles])

    @staticmethod
    def _text(article):
        # 문서명과 조 제목을 본문 앞에 붙인다. '위치정보'처럼 문서명에만 있는 말이
        # 질문에 나오면 그 문서 쪽 조항이 올라온다.
        return f"{article['document']} {article['title']} {article['text']}"

    def named_document(self, question):
        """질문이 약관 이름을 직접 부른 경우 그 이름을 돌려준다.

        '카카오 통합 약관'과 '카카오 통합서비스약관'이 섞이지 않도록 이름 전체로
        맞추고, 여러 개가 걸리면 가장 긴 이름을 쓴다.
        """
        text = re.sub(r"\s+", "", unicodedata.normalize("NFC", question))
        hits = [d for d in {a["document"] for a in self.articles}
                if re.sub(r"\s+", "", d) in text]
        return max(hits, key=len) if hits else None

    def apply_doc_boost(self, question, scores, doc_boost):
        """질문이 부른 약관 쪽을 끌어올린다. 잘라내지 않고 가산만 한다 —
        같은 내용이 여러 약관에 나란히 실린 문항이 실제로 있다."""
        named = self.named_document(question) if doc_boost else None
        if not named:
            return scores
        # 점수 폭을 기준으로 가산한다. max 만 쓰면 코사인 유사도처럼 음수가 나올 수
        # 있는 점수에서 가산점이 오히려 감점이 된다.
        bonus = doc_boost * (max(scores) - min(scores))
        return [s + bonus if a["document"] == named else s
                for s, a in zip(scores, self.articles)]

    def rank(self, question, doc_boost=0.5):
        """전체 조항을 점수 순으로 세운 인덱스 목록. 동점은 인덱스 순으로 고정한다."""
        scores = self.apply_doc_boost(
            question, self.bm25.scores(tokenize(question, **self.opts)), doc_boost
        )
        return sorted(range(len(self.articles)), key=lambda i: (-scores[i], i))

    def search(self, question, top_k=4, doc_boost=0.5):
        return [self.articles[i] for i in self.rank(question, doc_boost)[:top_k]]


BM25_INDEX = Index(ARTICLES, josa=False, bigram=True)
print("[인덱스] BM25 준비 완료")

# 임베딩은 BM25 가 놓치는 어휘 불일치 문항을 위한 보강이다. 없어도 결과기는 동작해야
# 하므로, 내려받기나 로드가 실패하면 BM25 단독으로 조용히 물러난다.
DENSE = None
try:
    _emb_tokenizer = AutoTokenizer.from_pretrained(EMB_MODEL_ID)
    _emb_model = AutoModel.from_pretrained(EMB_MODEL_ID, torch_dtype=DTYPE).to(DEVICE).eval()

    @torch.inference_mode()
    def _embed(texts, max_length=2048, batch_size=4):
        vectors = []
        for i in range(0, len(texts), batch_size):
            batch = _emb_tokenizer(
                texts[i:i + batch_size], padding=True, truncation=True,
                max_length=max_length, return_tensors="pt",
            ).to(_emb_model.device)
            hidden = _emb_model(**batch).last_hidden_state[:, 0]   # BGE 계열은 CLS 풀링
            vectors.append(torch.nn.functional.normalize(hidden, p=2, dim=-1))
        return torch.cat(vectors)

    _CHUNK_VECTORS = _embed([Index._text(a) for a in ARTICLES]).float().to("cpu")
    # 인덱싱이 끝나면 임베딩 모델을 CPU 로 내린다. 질의는 한 번에 한 문장이라
    # CPU 로도 금방 끝나고, 생성 모델에 VRAM 을 더 내줄 수 있다.
    _emb_model = _emb_model.float().to("cpu")
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    DENSE = (_embed, _CHUNK_VECTORS)
    print(f"[인덱스] 임베딩 준비 완료 {tuple(_CHUNK_VECTORS.shape)} (모델은 CPU 로 이동)")
except Exception as exc:                                          # noqa: BLE001
    print(f"[인덱스] 임베딩 사용 안 함 ({type(exc).__name__}: {exc}) — BM25 단독으로 진행")


def search(question, top_k=TOP_K, rrf_k=60):
    """BM25 와 임베딩의 '순위'를 융합한다.

    점수를 직접 더하지 않는 이유는 BM25 점수와 코사인 유사도가 스케일이 달라
    정규화 방식에 따라 결과가 요동치기 때문이다. RRF 는 그 문제를 피한다.
    """
    bm25_order = BM25_INDEX.rank(question, doc_boost=DOC_NAME_BOOST)
    if DENSE is None:
        order = bm25_order[:top_k]
    else:
        embed, vectors = DENSE
        scores = (vectors @ embed([question], max_length=512)[0].float()).tolist()
        scores = BM25_INDEX.apply_doc_boost(question, scores, DOC_NAME_BOOST)
        dense_order = sorted(range(len(ARTICLES)), key=lambda i: (-scores[i], i))
        fused = {}
        for ranking in (bm25_order, dense_order):
            for rank, idx in enumerate(ranking):
                fused[idx] = fused.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)
        # 동점은 인덱스 순으로 고정한다. 실행마다 결과가 달라지지 않게 한다.
        order = sorted(fused, key=lambda i: (-fused[i], i))[:top_k]
    return [ARTICLES[i] for i in order]


# ----- 1-3. 생성 모델 ----------------------------------------------------------------
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
if USE_7B:
    # 7B fp16 은 15GB 라 T4(16GB)에 들어가지 않는다. 4bit 로 올린다.
    _load_kwargs = dict(quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,   # T4 는 Turing 이라 bf16 미지원
    ))
else:
    _load_kwargs = dict(torch_dtype=DTYPE)

def _load_generator(model_id, load_kwargs):
    return AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=DEVICE,          # "auto" 를 쓰지 않는다. CPU 로 조용히 흘러가면 한 요청이 분 단위가 된다.
        attn_implementation="sdpa", # FlashAttention2 는 Ampere 이상 전용. T4 에서는 sdpa.
        **load_kwargs,
    ).eval()


try:
    gen_model = _load_generator(GEN_MODEL_ID, _load_kwargs)
except Exception as exc:                                          # noqa: BLE001
    # bitsandbytes 설치 실패나 VRAM 부족으로 4bit 로드가 깨져도 셀은 끝까지 실행돼야
    # 한다. 3B fp16 은 T4 에 확실히 들어간다.
    print(f"[모델] {GEN_MODEL_ID} 로드 실패 ({type(exc).__name__}: {exc}) — 3B fp16 으로 내려갑니다")
    GEN_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
    gen_model = _load_generator(GEN_MODEL_ID, dict(torch_dtype=DTYPE))
if gen_tokenizer.pad_token_id is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token


@torch.inference_mode()
def generate(messages, max_new_tokens):
    text = gen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = gen_tokenizer(text, return_tensors="pt").to(DEVICE)
    output = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,            # 그리디 — 실행마다 같은 답이 나온다
        repetition_penalty=1.05,
        pad_token_id=gen_tokenizer.pad_token_id,
    )
    out = gen_tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    del inputs, output
    if DEVICE == "cuda":
        torch.cuda.empty_cache()    # 문항 간 메모리 누적을 막는다
    return out


# ----- 1-4. 답변 조립 ----------------------------------------------------------------
import re


# "generate": 모델이 답을 쓴다(기본). "extract": 원문 구간을 그대로 이어 붙인다.
# 로컬 측정에서 extract 가 키팩트 F1 은 더 높다(0.7669 vs 0.6252). 판정 50점까지
# 이기는지는 연습 채점으로 확인해야 해서 기본값은 generate 로 둔다.
ANSWER_MODE = "generate"

CTX_FULL_RANKS = 2        # 상위 2개 조항은 길어도 통째로 넣는다
CTX_CHAR_BUDGET = 7000    # 3·4위 조항 본문은 이 예산 안에서만 넣는다

# 유형별 생성 토큰 상한. 이건 목표 길이가 아니라 '잘리지 않게 하는 상한'이다.
# 답을 짧게 만드는 일은 프롬프트가 하고, 이 값은 열거형이 중간에 끊기는 것과
# 한 요청이 러너의 120초 제한에 걸리는 것만 막는다. 그래서 넉넉한 쪽으로 잡는다.
TOKEN_BUDGET = {"yesno": 256, "single": 288, "multi": 384, "enum": 512}

# '몇 가지/몇 종류'는 열거지만 '몇 명/몇 개월'은 아니다. 숫자+가지 표현도 함께 본다.
_ENUM_RE = re.compile(
    r"[0-9]+\s*가지|몇\s*가지|몇\s*종류|"
    r"[한두세네다섯여섯일곱여덟아홉열일이삼사오육칠팔구십]\s*가지|"
    r"열거|나열|각각\s*무엇|모두\s*무엇|어떤\s*것들|종류.{0,8}무엇|항목.{0,8}무엇")
_YESNO_RE = re.compile(r"(우선하여|우선\s*적용|허용되나요|허용됩니까|가능한가요|맞나요|"
                       r"되나요|인가요|있나요|끝나나요|하나요|합니까)\s*\??\s*$")
_WH_RE = re.compile(r"무엇|어떤|누가|어디|언제|얼마|며칠|몇|어떻게|왜")


def classify(question):
    """생성 예산을 정하기 위한 질문 유형. 모델을 한 번 더 부르지 않는다."""
    text = question.strip()
    if _ENUM_RE.search(text):
        return "enum"
    wh = len(set(_WH_RE.findall(text)))
    if wh >= 2:
        return "multi"
    if _YESNO_RE.search(text) and wh == 0:
        return "yesno"
    return "single"


def build_context(hits, full_ranks=CTX_FULL_RANKS, budget=CTX_CHAR_BUDGET):
    """상위 조항은 통째로, 나머지는 예산 안에서 넣는다.

    상위 조항을 글자 수로 자르지 않는 것이 중요하다. 열거형 문항의 정답은 조항
    뒷부분에 있는 경우가 많아, 자르면 근거 목록에는 남고 답에서는 사라진다.
    """
    blocks, used = [], 0
    for i, hit in enumerate(hits, 1):
        head = f"[{hit['document']} 제{hit['article']}조({hit['title']})]"
        body = hit["text"]
        if i <= full_ranks or used + len(body) <= budget:
            used += len(body)
            blocks.append(f"<조항 {i}> {head}\n{body}")
        else:
            blocks.append(f"<조항 {i}> {head}")
    return "\n\n".join(blocks)


SYSTEM_PROMPT = (
    "제시된 카카오 약관 조항만을 근거로 질문에 답하세요.\n"
    "\n"
    "1. 한국어로만 답합니다. 중국어·일본어를 섞지 않고, 번역하거나 설명을 덧붙이겠다는 "
    "말도 쓰지 않으며, 답변 본문만 한 번 출력합니다.\n"
    "2. 근거가 되는 문장은 요약하거나 바꿔 쓰지 말고 조항의 표현과 어순을 그대로 옮깁니다. "
    "숫자·기간·비율·법령명·조문번호·회사명·서비스명은 한 글자도 바꾸지 않습니다. "
    "'1인'을 '1명'으로, '금지됩니다'를 '허용되지 않습니다'로 바꾸지 않습니다.\n"
    "3. 조항에 없는 낱말을 새로 만들어 쓰지 않습니다. 배경 설명, 요약 문장, 소감, "
    "'위 조항에 따르면' 같은 군더더기를 붙이지 않고 곧바로 답만 씁니다.\n"
    "4. 질문이 여러 가지를 물으면 각각에 모두 답합니다. 무엇을·어떤 방법으로·어디에·"
    "어떤 효력을 처럼 나뉘어 있으면 하나도 빠뜨리지 않습니다.\n"
    "5. 열거를 묻는 질문이면 항목 이름만 쓰지 말고 조항에 적힌 각 항목의 설명까지 함께, "
    "조항이 밝힌 개수만큼 빠짐없이 옮깁니다.\n"
    "6. 질문의 전제를 그대로 믿지 마십시오. 조항이 전제와 다르면 '아니오'로 시작해 "
    "바로잡습니다. 다만 '예/아니오'는 가부를 묻는 질문에만 쓰고, 무엇·어떻게·얼마처럼 "
    "설명을 요구하는 질문에는 곧바로 내용을 답합니다.\n"
    "7. 제시된 조항에 근거가 조금이라도 있으면 반드시 그 내용으로 답합니다. "
    "'확인되지 않습니다', '알 수 없습니다' 라고 답하지 않습니다."
)

# 실제 약관에 없는 가상의 조항·회사명만 쓴다. 진짜 조문 번호나 회사명을 예시에
# 넣으면 그 값이 다른 문항의 답변으로 새어 나온다.
FEWSHOT = [
    {"role": "user", "content": (
        "다음은 관련 약관 조항입니다.\n\n"
        "<조항 1> [예시 약관 제1조(준칙)]\n"
        "본 약관에 규정되지 않은 사항은 세부지침에 따릅니다. 본 약관과 세부지침의 내용이 "
        "충돌할 경우 세부지침에 따릅니다.\n\n"
        "질문: 본 약관이 세부지침보다 우선하여 적용되나요?\n\n답변:")},
    {"role": "assistant", "content":
        "아니오. 본 약관과 세부지침의 내용이 충돌할 경우 세부지침에 따릅니다."},
    {"role": "user", "content": (
        "다음은 관련 약관 조항입니다.\n\n"
        "<조항 1> [예시 약관 제2조(대상 서비스)]\n"
        "서비스 명칭에 '예시'가 사용되더라도 회사가 아닌 예시 계열사에서 제공하는 서비스"
        "(예: ㈜가나다물류가 제공하는 가나다 배송 서비스)는 본 약관의 대상 서비스에 "
        "포함되지 않습니다.\n\n"
        "질문: 본 약관의 대상 서비스에 포함되지 않는 서비스는 누가 제공하는 서비스이며, "
        "약관은 그 예로 무엇을 들고 있나요?\n\n답변:")},
    {"role": "assistant", "content": (
        "회사가 아닌 예시 계열사에서 제공하는 서비스입니다. 약관은 그 예로 ㈜가나다물류가 "
        "제공하는 가나다 배송 서비스를 들고 있습니다.")},
]


def build_messages(question, hits):
    return (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + FEWSHOT
        + [{"role": "user", "content":
            f"다음은 관련 약관 조항입니다.\n\n{build_context(hits)}\n\n"
            f"질문: {question}\n\n답변:"}]
    )


_PREAMBLE_RE = re.compile(r"^\s*(답변\s*[:：]|답\s*[:：]|정답\s*[:：])\s*")
_FILLER_RE = re.compile(
    r"^\s*(?:위\s*조항에\s*따르면|제시된\s*조항에\s*따르면|해당\s*조항에\s*따르면|"
    r"약관에\s*따르면)\s*[,，]?\s*")
# 가운뎃점 ・(U+30FB)는 빼야 한다. 약관의 "변경・제한・중지" 에서 이걸 지우면
# 세 낱말이 한 낱말로 붙어 채점 토큰 세 개를 한꺼번에 잃는다.
_CJK_RE = re.compile(r"[一-鿿぀-ゟァ-ヺ]")


def postprocess(text):
    """새 낱말만 늘리는 군더더기를 걷어낸다. F1 분모가 그만큼 줄어든다."""
    out = _PREAMBLE_RE.sub("", text.strip())
    out = _FILLER_RE.sub("", out)
    out = _CJK_RE.sub("", out)                      # 한자·가나 혼입 제거
    # 같은 문장을 두 번 쓰면 판정의 명료성이 깎인다. 순서는 유지하고 중복만 없앤다.
    seen, kept = set(), []
    for part in re.split(r"(?<=다\.)\s+|\n+", out):
        key = re.sub(r"\s+", "", part)
        if key and key in seen:
            continue
        seen.add(key)
        kept.append(part.strip())
    return re.sub(r"[ \t]+", " ", "\n".join(k for k in kept if k)).strip()


_REFUSAL_RE = re.compile(
    r"확인되지\s*않|확인할\s*수\s*없|찾을\s*수\s*없|알\s*수\s*없|"
    r"명시되어\s*있지\s*않|나와\s*있지\s*않|포함되어\s*있지\s*않|정보가\s*없")


# 조항을 항(①②) · 호(1. 2.) 단위로 먼저 자른다. 열거형 문항의 정답은 호 하나가
# 통째로 한 항목이라, 문장으로 쪼개면 항목 이름과 설명이 흩어진다.
_SPAN_SPLIT_RE = re.compile(r"(?=(?:^|\s)(?:\d{1,2}\.\s|[①-⑳]))")
_SPAN_MAX_CHARS = 260


def source_spans(text):
    parts = [p.strip() for p in _SPAN_SPLIT_RE.split(text) if p.strip()]
    spans = []
    for part in parts:
        if len(part) <= _SPAN_MAX_CHARS:
            spans.append(part)
        else:
            spans.extend(s.strip() for s in re.split(r"(?<=다\.)\s+", part) if s.strip())
    # 항 기호(①②)는 채점 토큰이 아니라 떼어내도 F1 이 변하지 않는다. 답변으로 읽힐 때
    # 조문을 그대로 퍼온 티가 덜 나서 판정의 명료성에만 도움이 된다.
    return [re.sub(r"^[①-⑳]\s*", "", s).strip() for s in spans]


def extractive_answer(question, hits, kind="single"):
    """검색된 조항에서 질문과 겹치는 구간을 원문 그대로 골라 잇는다.

    공개 10문항 기준 키팩트 F1 0.7669 (23.0/30점) — 같은 문항에서 생성 답변
    최고치 0.6252 보다 높다. 표현을 한 글자도 바꾸지 않으니 당연한 결과다.
    다만 판정(50점)의 명료성·완결성까지 이기는지는 확인되지 않아 기본값이 아니다.
    회피가 나왔을 때의 안전망으로는 무조건 이쪽이 낫다.
    """
    if kind == "enum":
        # 열거형은 항목을 고르는 순간 지는 문제다. 5개 중 3개만 집으면 나머지 2개의
        # 낱말을 통째로 잃는다. 1위 조항을 통째로 옮기는 편이 낫다 —
        # 공개 10문항에서 P03 이 0.194 -> 0.718, 전체 0.7145 -> 0.7669.
        return " ".join(source_spans(hits[0]["text"]))

    query = set(tokenize(question))
    n_spans = 1
    candidates = []
    for rank, hit in enumerate(hits):
        for order, span in enumerate(source_spans(hit["text"])):
            candidates.append((len(query & set(tokenize(span))), rank, order, span))
    if not candidates:
        return hits[0]["text"]
    candidates.sort(key=lambda c: (-c[0], c[1], c[2]))
    picked = [c for c in candidates[:n_spans] if c[0] > 0] or candidates[:1]
    return " ".join(c[3] for c in sorted(picked, key=lambda c: (c[1], c[2])))


def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점.

    반환 형식은 계약이다:
        {"answer": "...", "retrieved": [["카카오계정 약관", 3], ...]}   # 1~4개
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    question = question.strip()

    hits = search(question, top_k=TOP_K)                              # noqa: F821
    kind = classify(question)
    if ANSWER_MODE == "extract":
        answer = extractive_answer(question, hits, kind)
    else:
        answer = postprocess(generate(build_messages(question, hits),  # noqa: F821
                                      TOKEN_BUDGET[kind]))
        # 근거가 있는데 회피하면 판정이 100점 만점에 15점으로 떨어진다.
        if not answer or _REFUSAL_RE.search(answer):
            answer = extractive_answer(question, hits, kind)

    # 프롬프트에 넣은 조항을 검색 순위 그대로 적는다. 빈 목록은 계약 위반이다.
    retrieved, seen = [], set()
    for hit in hits:
        key = (hit["document"], hit["article"])
        if key not in seen:
            seen.add(key)
            retrieved.append([hit["document"], hit["article"]])
    return {"answer": answer, "retrieved": retrieved[:TOP_K]}         # noqa: F821


# 워밍업 — CUDA 커널 초기화 비용을 첫 채점 요청이 아니라 여기서 태운다.
_warm = answer_question("이 약관의 목적은 무엇인가요?")
assert isinstance(_warm["answer"], str) and 1 <= len(_warm["retrieved"]) <= 4
print(f"[모델] {GEN_MODEL_ID} 로드 및 워밍업 완료 — 근거 {_warm['retrieved']}")


# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app 을 localhost 에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock 을 사용합니다.
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 2번 공통 러너를 실행하세요.")


[인덱스] 조항 72개 — 카카오계정 약관 17 / 카카오 위치정보 이용약관 16 / 카카오 통합서비스약관 18 / 카카오 통합 약관 21
[인덱스] BM25 준비 완료


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[모델] Qwen/Qwen2.5-7B-Instruct 로드 및 워밍업 완료 — 근거 [['카카오 통합서비스약관', 1], ['카카오 통합서비스약관', 2], ['카카오 통합 약관', 2], ['카카오 위치정보 이용약관', 2]]
[1번 셀 준비] 2번 공통 러너를 실행하세요.


In [2]:
# 2번 셀 — 공개 10문항 답변 파일 생성
# 이 셀은 전 팀 공통이며 _SP_TEAM 한 줄 외에는 수정하지 않습니다.
# 새 Google Colab T4 런타임에서 결과기 코드를 먼저 실행한 뒤 이 셀을 실행합니다.
#
# 사용 순서
# 1. 새 Google Colab T4 런타임에서 1번 셀 결과기 코드를 실행합니다.
# 2. 이 공통 러너를 2번 셀에 그대로 둡니다.
# 3. 맨 위 _SP_TEAM에 운영진이 알려준 숫자 팀 식별자를 입력합니다.
# 4. 생성된 answers_public_<팀>.json을 결과기 코랩 파일과 함께 제출합니다.
# 공개 문항 10개 · 실행 방식: http
# ═══════════════════════════════════════════════════════════════
#  ★ 여기 한 줄만 자기 팀으로 바꾸세요. 나머지는 손대지 마세요. ★
# ═══════════════════════════════════════════════════════════════
_SP_TEAM = "9"          # 예: "1"  ← 운영진이 알려준 팀 식별자(숫자)를 그대로 적습니다
# ═══════════════════════════════════════════════════════════════

import builtins as _sp_builtins
import json as _sp_json
import os as _sp_os_rt
import re as _sp_re
import signal as _sp_signal
import socket as _sp_socket
import sys as _sp_sys
import time as _sp_time
import traceback as _sp_traceback
import unicodedata as _sp_unicodedata
import urllib.error as _sp_urlerror
import urllib.request as _sp_urlrequest

_sp_open = _sp_builtins.open
_sp_print = _sp_builtins.print

if "_sp_real_sys_exit" in globals():
    _sp_sys.exit = _sp_real_sys_exit
    if _sp_real_exit is not None:
        _sp_builtins.exit = _sp_real_exit
    if _sp_real_quit is not None:
        _sp_builtins.quit = _sp_real_quit

_SP_OUTPUT_DIR = "/content/"
_SP_OUTPUT_PREFIX = "answers_public_"
_SP_EXPECTED_OUTPUT_PATH = ""
_SP_TEAM_ALLOWED = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz_-"
_SP_TEAM_MAX_LEN = 32
_SP_TEAM_NUMERIC_ONLY = True

def _sp_team_howto(head):
    """중단 사유 + 학생이 바로 고칠 수 있는 안내를 한 덩어리로 만든다."""
    rule = (
        "1 이상의 정수를 문자열로 입력합니다. 예: 1, 2, 17"
        if _SP_TEAM_NUMERIC_ONLY
        else "영문·숫자·밑줄(_)·하이픈(-) 1~" + str(_SP_TEAM_MAX_LEN) + "자"
    )
    return (
        head
        + "\n"
        + "\n  [고치는 법] 이 셀 맨 위 ★ 상자 안의 한 줄을 이렇게 바꾸세요."
        + '\n      _SP_TEAM = "1"      ← 운영진이 알려준 팀 식별자(숫자)를 따옴표 안에 그대로'
        + "\n  [쓸 수 있는 값] " + rule
        + "\n                 띄어쓰기와 / \\ . : 같은 경로 문자는 파일 이름을 깨뜨려 쓸 수 없습니다."
        + "\n  [왜] 결과 파일 이름이 " + _SP_OUTPUT_PREFIX + "<팀>.json 이고, 채점은 이 이름으로"
        + "\n       어느 팀 답안인지 가립니다. 비워 두면 채점 자체가 되지 않습니다."
    )

def _sp_resolve_team(value):
    """_SP_TEAM 을 검사·정리해 돌려준다. 쓸 수 없는 값이면 RuntimeError 로 즉시 중단."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 비어 있어 실행을 중단했습니다. 결과 파일은 만들지 않았습니다."))
    team = value.strip()
    if _SP_TEAM_NUMERIC_ONLY and not _sp_re.fullmatch(r"[1-9][0-9]*", team):
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)는 운영진이 알려준 숫자여야 합니다. 지금 값: " + repr(value)))
    if len(team) > _SP_TEAM_MAX_LEN:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 너무 깁니다(" + str(len(team)) + "자). 팀 이름이 아니라 짧은 식별자입니다."))
    _bad = _sp_builtins.sorted(
        _sp_builtins.set(c for c in team if c not in _SP_TEAM_ALLOWED and not ("가" <= c <= "힣")))
    if _bad:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)에 파일 이름으로 쓸 수 없는 문자가 있습니다: "
            + ", ".join(repr(c) for c in _bad) + "   (지금 값: " + repr(value) + ")"))
    return team

_SP_TEAM = _sp_resolve_team(_SP_TEAM)
if any(ord(c) > 127 for c in _SP_TEAM):
    _sp_print("[주의] 팀 식별자에 한글 등 ASCII 밖 문자가 있습니다: " + _SP_TEAM
              + " — 운영진이 알려준 식별자가 맞는지 확인하세요."
              " 한글 파일 이름은 내려받기·올리기 과정에서 자모 표현이 달라져 팀이 어긋날 수 있습니다.",
              flush=True)

_SP_OUTPUT_PATH = _SP_OUTPUT_DIR.rstrip("/") + "/" + _SP_OUTPUT_PREFIX + _SP_TEAM + ".json"
if _SP_EXPECTED_OUTPUT_PATH and (_sp_os_rt.path.basename(_SP_OUTPUT_PATH)
                                 != _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH)):
    raise RuntimeError(
        "이 셀은 " + _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH) + " 용으로 생성됐는데 "
        + _sp_os_rt.path.basename(_SP_OUTPUT_PATH) + " 로 저장하려 합니다"
        "(_SP_TEAM 을 손으로 고쳤습니까?). 다른 팀으로 돌리려면 --team 을 바꿔 셀을 다시 생성하세요."
    )
_SP_AUTO_DOWNLOAD = True
_SP_QUESTIONS_JSON = (
    "[[\"P01\", \"사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있으며, 다른 사람과 공유하는 것은 허용되나요?\"], [\"P02\", \"회사가 예측하거나 통제할 수 없는 사유로 서비스가 중단된 경우, 복구가 몇 시간 이상 지연되면 회사는 공지사항에 게시하여 알리나요?\"], [\"P03\", \"카카오계정 약관에서 회사가 개별 서비스와 연동하여 카카오계정에서 제공한다고 열거한 '카카오계정 서비스'의 내용 5가지는 각각 무엇인가요?\"], [\"P04\", \"회사가 위치기반서비스의 이용을 제한하거나 중지한 때에는 이용자에게 무엇을 어떤 방법으로 알리나요?\"], [\"P05\", \"회사가 위치정보 수집·이용·제공사실 확인자료를 기록·보존하는 근거는 위치정보의 보호 및 이용 등에 관한 법률 제 몇 조 제 몇 항이며, 그 자료는 어디에 기록되어 몇 개월간 보관되나요?\"], [\"P06\", \"카카오계정이 없는 사람이 통합서비스에 가입하려면 무엇을 먼저 해야 하며, 통합서비스 이용계약은 동의·확인·승낙의 어떤 순서로 체결되나요?\"], [\"P07\", \"서비스 명칭에 '카카오'가 사용되더라도 카카오 통합서비스약관의 '통합서비스'에 포함되지 않는 서비스는 누가 제공하는 서비스이며, 약관은 그 예로 무엇을 들고 있나요?\"], [\"P08\", \"카카오 통합 약관과 세부지침(회사가 정한 서비스의 개별 이용약관·운영정책·규칙 등)의 내용이 충돌하는 경우"
    ", 본 약관이 세부지침보다 우선하여 적용되나요?\"], [\"P09\", \"이용자가 서비스 사용을 중단하거나 카카오계정 및 Daum 아이디를 탈퇴한 이후, 게시물에 관하여 회사에 부여한 라이선스의 효력은 어떻게 되나요?\"], [\"P10\", \"8세 이하의 아동 등의 생명 또는 신체 보호를 위해 보호의무자가 개인위치정보의 이용 또는 제공에 동의하려면 어떤 서류에 무엇을 첨부하여 어디에 제출해야 하며, 그 동의는 어떤 효력을 갖나요?\"]]"
)
_SP_QUESTIONS = [tuple(_x) for _x in _sp_json.loads(_SP_QUESTIONS_JSON)]
_SP_ALLOWED_DOCS = _sp_json.loads("[\"카카오계정 약관\", \"카카오 통합서비스약관\", \"카카오 통합 약관\", \"카카오 위치정보 이용약관\"]")
_SP_PER_Q_TIMEOUT_S = 120
_SP_TRANSPORT = "http"
_SP_HTTP_HOST = "127.0.0.1"
_SP_HTTP_PORT = 8765
_SP_HTTP_STARTUP_TIMEOUT_S = 30
_SP_HTTP_HEALTH_PATH = "/health"
_SP_HTTP_ANSWER_PATH = "/answer"
_SP_PERFORMANCE_REQUESTS = 12
_SP_PERFORMANCE_CONCURRENCY = 2
_SP_PERFORMANCE_REPETITIONS = 3
_SP_PERFORMANCE_WARMUP_REQUESTS = 2

_sp_fn = globals().get("answer_question")
if not callable(_sp_fn):
    raise RuntimeError(
        "팀 코드에 answer_question(question) 함수가 없습니다(규정 ②). 실행을 중단합니다."
    )

_sp_doc_warnings = []
_sp_timeouts = []
_sp_http_server = None
_sp_http_thread = None

class _SpHttpTimeout(Exception):
    """HTTP 요청 시간 초과. 품질 추출에서는 timeout_qids로 기록한다."""

def _sp_http_url(path):
    return "http://" + _SP_HTTP_HOST + ":" + str(_SP_HTTP_PORT) + path

def _sp_http_json(method, path, payload=None, timeout_s=None):
    data = None
    headers = {"Accept": "application/json"}
    if payload is not None:
        data = _sp_json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = _sp_urlrequest.Request(
        _sp_http_url(path), data=data, headers=headers, method=method
    )
    try:
        with _sp_urlrequest.urlopen(req, timeout=timeout_s or _SP_PER_Q_TIMEOUT_S) as resp:
            raw = resp.read().decode("utf-8")
            if resp.status != 200:
                raise RuntimeError("HTTP " + str(resp.status) + ": " + raw[:500])
    except (_sp_socket.timeout, TimeoutError) as exc:
        raise _SpHttpTimeout(str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다.") from exc
    except _sp_urlerror.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError("HTTP " + str(exc.code) + ": " + raw[:500]) from exc
    except _sp_urlerror.URLError as exc:
        if isinstance(exc.reason, (_sp_socket.timeout, TimeoutError)):
            raise _SpHttpTimeout(
                str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다."
            ) from exc
        raise RuntimeError("HTTP 연결 실패: " + str(exc.reason)) from exc
    try:
        return _sp_json.loads(raw)
    except _sp_json.JSONDecodeError as exc:
        raise TypeError("HTTP 응답이 JSON이 아닙니다: " + raw[:500]) from exc

def _sp_start_http_server():
    global _sp_http_server, _sp_http_thread
    _sp_app = globals().get("app")
    if _sp_app is None:
        raise RuntimeError(
            "HTTP 실행 모드에는 전역 FastAPI app과 GET /health, POST /answer가 필요합니다."
        )
    try:
        import threading as _sp_threading
        import uvicorn as _sp_uvicorn
    except ImportError as exc:
        raise RuntimeError(
            "HTTP 실행 모드에는 fastapi와 uvicorn이 필요합니다. 팀 설치 목록에 추가하세요."
        ) from exc
    _sp_config = _sp_uvicorn.Config(
        _sp_app,
        host=_SP_HTTP_HOST,
        port=_SP_HTTP_PORT,
        workers=1,
        log_level="warning",
        access_log=False,
    )
    _sp_http_server = _sp_uvicorn.Server(_sp_config)
    _sp_http_thread = _sp_threading.Thread(
        target=_sp_http_server.run, name="ktb-fastapi", daemon=True
    )
    _sp_http_thread.start()
    _sp_deadline = _sp_time.time() + _SP_HTTP_STARTUP_TIMEOUT_S
    _sp_last = None
    while _sp_time.time() < _sp_deadline:
        if not _sp_http_thread.is_alive():
            raise RuntimeError("FastAPI 서버가 준비되기 전에 종료됐습니다.")
        try:
            health = _sp_http_json("GET", _SP_HTTP_HEALTH_PATH, timeout_s=1)
            if isinstance(health, dict):
                _sp_print("[서버] FastAPI /health 준비 완료: " + _sp_http_url(_SP_HTTP_HEALTH_PATH))
                return
        except Exception as exc:
            _sp_last = exc
        _sp_time.sleep(0.2)
    _sp_stop_http_server()
    raise RuntimeError(
        "FastAPI 서버가 " + str(_SP_HTTP_STARTUP_TIMEOUT_S)
        + "초 안에 준비되지 않았습니다: " + str(_sp_last)
    )

def _sp_stop_http_server():
    if _sp_http_server is not None:
        _sp_http_server.should_exit = True
    if _sp_http_thread is not None and _sp_http_thread.is_alive():
        _sp_http_thread.join(timeout=5)

def _sp_invoke(question):
    if _SP_TRANSPORT == "http":
        return _sp_http_json(
            "POST", _SP_HTTP_ANSWER_PATH, {"question": question},
            timeout_s=_SP_PER_Q_TIMEOUT_S,
        )
    return _sp_call_with_timeout(_sp_fn, question, _SP_PER_Q_TIMEOUT_S)

if _SP_TRANSPORT == "http":
    _sp_start_http_server()

_sp_env_warnings = []
for _sp_d in ("/content/drive", "/content/gdrive", "/gdrive"):
    if _sp_os_rt.path.ismount(_sp_d):
        _sp_env_warnings.append(_sp_d + " 가 마운트되어 있습니다")
if _sp_env_warnings:
    _sp_print("", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("[규정 ③ 경고] 이 세션은 운영진 실행 환경과 다릅니다.", flush=True)
    for _sp_w in _sp_env_warnings:
        _sp_print("  · " + _sp_w, flush=True)
    _sp_print("  운영진은 드라이브가 연결되지 않은 새 세션에서 실행합니다. 드라이브에 둔 약관·인덱스를", flush=True)
    _sp_print("  읽고 있다면 본선에서 전량 실패합니다. 약관은 실행 중 내려받거나 셀 안에 포함하세요.", flush=True)
    _sp_print("  확인 방법: 새 노트북을 열어 코드와 이 셀만 붙여 넣고 실행해 보세요.", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("", flush=True)

class _SpTimeout(BaseException):
    """문항 단위 시간 초과.

    **BaseException 을 상속하는 것이 핵심이다.** 팀 코드가 `try/except Exception` 으로
    넓게 감싸는 일은 흔한데, Exception 을 상속하면 그 handler 가 시간 초과를 삼켜
    상한이 무력화된다(그대로 다음 루프를 돌며 계속 매달린다).
    """

def _sp_call_with_timeout(fn, arg, seconds):
    """SIGALRM 으로 문항 호출에 상한을 건다.

    메인 스레드가 아니거나 SIGALRM 이 없는 환경(윈도 등)에서는 signal 설정이
    실패하므로, 그때는 상한 없이 그대로 호출한다 — 상한을 못 걸었다고 해서
    채점 자체를 포기하는 편이 더 나쁘다.

    웹 Colab 셀은 IPython 이 메인 스레드에서 실행하므로 정상 동작한다.
    """
    if not seconds or seconds <= 0:
        return fn(arg)
    _sp_secs = max(1, int(seconds))     # alarm() 은 정수만 받는다. 0 은 '취소' 라 최소 1초.

    def _sp_on_alarm(signum, frame):
        raise _SpTimeout(str(_sp_secs) + "초 안에 응답하지 않았습니다.")

    try:
        _sp_prev = _sp_signal.signal(_sp_signal.SIGALRM, _sp_on_alarm)
        _sp_signal.alarm(_sp_secs)
    except (ValueError, AttributeError, OSError):
        return fn(arg)          # 상한을 걸 수 없는 환경 — 그대로 실행
    try:
        return fn(arg)
    finally:
        _sp_signal.alarm(0)
        try:
            _sp_signal.signal(_sp_signal.SIGALRM, _sp_prev)
        except Exception:
            pass

def _sp_json_safe_art(art):
    """조번호를 JSON 으로 쓸 수 있는 값으로. 표기는 최대한 원본을 살린다.

    **여기서 흡수하지 않으면 30문항을 다 돌린 뒤 파일 저장에서 터진다.**
    일부 수치 라이브러리의 정수형은 dict 도 아니고 2원소 검사도 통과하지만
    json.dump 가 거부한다. 이 값을 흡수하지 않으면
    실패 시점이 맨 끝이라 GPU 시간을 다 쓰고 결과 파일이 없는 최악의 형태가 된다.

    '제7조' 같은 문자열은 그대로 둔다 — 채점기 _art_no 가 정수로 읽는다.
    """
    if isinstance(art, bool):        # bool 은 int 의 하위형이라 먼저 걸러 낸다
        return str(art)
    if isinstance(art, (int, str)):
        return art
    try:                              # np.int64 등 정수로 볼 수 있는 것
        return int(art)
    except (TypeError, ValueError):
        return str(art)

def _sp_norm_doc(x):
    """문서명 대조용 정규화 — NFC 통일 + 공백 전부 제거.

    ⚠️ 채점기 judge_service/engine/objective.py 의 `norm_doc` 과 **같은 규칙이어야 한다.**
    러너는 Colab 셀이라 judge_service 를 import 할 수 없어 규칙을 여기에 복제해 둔다.
    한쪽만 바뀌어 어긋나면 곧바로 오탐이 난다 — 예전에 러너가 완전 일치로 대조하던 때
    '카카오계정약관'·'카카오 계정 약관' 은 실제 채점 MRR 이 1.00 인데도 규정 ④ 위반 경고를
    맞았다. 팀은 없는 문제를 고치러 다니고(자가 확인표가 n_doc_violations == 0 을 요구한다),
    정상 팀이 경고를 맞기 시작하면 아무도 경고를 안 보게 된다.
    두 구현의 일치는 submission_pipeline/tests/test_doc_name_normalization.py 가 고정한다.
    """
    return _sp_re.sub(r"\s+", "", _sp_unicodedata.normalize("NFC", str(x)))

_SP_ALLOWED_DOCS_NORM = _sp_builtins.set(_sp_norm_doc(_d) for _d in _SP_ALLOWED_DOCS)

def _sp_normalize_retrieved(qid, value):
    """retrieved 를 근거순 [[문서명, 조번호], ...] 1~4개로 정규화."""
    if not isinstance(value, (list, tuple)):
        raise TypeError(qid + ": retrieved 는 목록이어야 합니다. (실제: " + type(value).__name__ + ")")
    out = []
    for item in value:
        if isinstance(item, dict) and "doc" in item and "article_no" in item:
            doc, art = item["doc"], item["article_no"]
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            doc, art = item
        else:
            raise TypeError(qid + ": retrieved 항목은 [문서명, 조번호] 2원소여야 합니다. (실제: " + repr(item) + ")")
        doc = str(doc)
        if _SP_ALLOWED_DOCS_NORM and _sp_norm_doc(doc) not in _SP_ALLOWED_DOCS_NORM:
            _sp_doc_warnings.append({"qid": qid, "doc": doc})
        out.append([doc, _sp_json_safe_art(art)])
    if not 1 <= len(out) <= 4:
        raise ValueError(
            qid + ": retrieved 는 실제 답변 근거를 관련도 순으로 1~4개 반환해야 합니다. "
            "(실제: " + str(len(out)) + "개)"
        )
    return out

_sp_answers = []
_sp_errors = []
_sp_total = len(_SP_QUESTIONS)
_sp_print(
    "\n========== " + "공개" + " " + str(_sp_total)
    + "문항 실행 · " + _SP_TEAM + "팀 ==========",
    flush=True,
)
_sp_t0 = _sp_time.time()

for _sp_i, (_sp_qid, _sp_q) in enumerate(_SP_QUESTIONS, 1):
    _sp_print("[" + str(_sp_i).zfill(2) + "/" + str(_sp_total) + "] " + _sp_qid + " 실행 중 ...", flush=True)
    _sp_started = _sp_time.time()
    try:
        _sp_out = _sp_invoke(_sp_q)
        if not isinstance(_sp_out, dict):
            raise TypeError(_sp_qid + ": answer_question() 은 딕셔너리를 반환해야 합니다. (실제: "
                            + type(_sp_out).__name__ + ")")
        _sp_retrieved = _sp_normalize_retrieved(_sp_qid, _sp_out.get("retrieved"))
        _sp_answer = _sp_out.get("answer")
        if not isinstance(_sp_answer, str):
            raise TypeError(_sp_qid + ": answer 는 문자열이어야 합니다. (실제: "
                            + type(_sp_answer).__name__ + ")")
        _sp_answers.append({"qid": _sp_qid, "retrieved": _sp_retrieved, "answer": _sp_answer})
    except (_SpTimeout, _SpHttpTimeout) as _sp_exc:  # 한 문항이 세션 전체를 잡아먹지 않도록 끊는다.
        _sp_msg = "Timeout: " + str(_sp_exc)
        _sp_timeouts.append(_sp_qid)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[시간초과] " + _sp_qid + " — " + _sp_msg, flush=True)
    except Exception as _sp_exc:  # 한 문항 실패로 30문항 전체를 잃지 않는다.
        _sp_msg = type(_sp_exc).__name__ + ": " + str(_sp_exc)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[오류] " + _sp_qid + " — " + _sp_msg, flush=True)
        _sp_traceback.print_exc()
    finally:
        _sp_print("      (" + str(round(_sp_time.time() - _sp_started, 1)) + "s)", flush=True)

_sp_performance = None
if _SP_TRANSPORT == "http" and _SP_PERFORMANCE_REQUESTS > 0:
    from concurrent.futures import ThreadPoolExecutor as _SpThreadPoolExecutor

    def _sp_perf_one(index):
        _qid, _question = _SP_QUESTIONS[index % len(_SP_QUESTIONS)]
        started = _sp_time.perf_counter()
        try:
            value = _sp_http_json(
                "POST", _SP_HTTP_ANSWER_PATH, {"question": _question},
                timeout_s=_SP_PER_Q_TIMEOUT_S,
            )
            ok = (
                isinstance(value, dict)
                and isinstance(value.get("answer"), str)
                and isinstance(value.get("retrieved"), (list, tuple))
            )
            return {
                "ok": ok,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": None if ok else "invalid_schema",
            }
        except Exception as exc:
            return {
                "ok": False,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": type(exc).__name__ + ": " + str(exc),
            }

    def _sp_percentile(values, ratio):
        if not values:
            return None
        pos = min(len(values) - 1, max(0, int((len(values) - 1) * ratio)))
        return round(values[pos], 4)

    def _sp_median(values):
        values = sorted(values)
        if not values:
            return None
        middle = len(values) // 2
        if len(values) % 2:
            return values[middle]
        return (values[middle - 1] + values[middle]) / 2

    def _sp_perf_round(n_requests, repetition):
        started = _sp_time.perf_counter()
        with _SpThreadPoolExecutor(max_workers=max(1, _SP_PERFORMANCE_CONCURRENCY)) as pool:
            rows = list(pool.map(_sp_perf_one, range(n_requests)))
        wall_s = _sp_time.perf_counter() - started
        ok_rows = [row for row in rows if row["ok"]]
        latencies = sorted(row["latency_s"] for row in ok_rows)
        return {
            "repetition": repetition,
            "transport": "http",
            "requests": n_requests,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "success": len(ok_rows),
            "fail": len(rows) - len(ok_rows),
            "success_rate": round(len(ok_rows) / len(rows), 4),
            "throughput_rps": round(len(ok_rows) / wall_s, 4) if wall_s else 0.0,
            "wall_s": round(wall_s, 4),
            "p50_latency_s": _sp_percentile(latencies, 0.50),
            "p95_latency_s": _sp_percentile(latencies, 0.95),
            "errors": [row for row in rows if not row["ok"]],
        }

    _sp_warmup = None
    if _SP_PERFORMANCE_WARMUP_REQUESTS > 0:
        _sp_print(
            "[성능] 워밍업 " + str(_SP_PERFORMANCE_WARMUP_REQUESTS) + "요청 실행 중 ...",
            flush=True,
        )
        _sp_warmup = _sp_perf_round(_SP_PERFORMANCE_WARMUP_REQUESTS, 0)

    _sp_perf_samples = []
    for _sp_repetition in range(1, _SP_PERFORMANCE_REPETITIONS + 1):
        _sp_print(
            "[성능] 측정 " + str(_sp_repetition) + "/"
            + str(_SP_PERFORMANCE_REPETITIONS) + " 실행 중 ...",
            flush=True,
        )
        _sp_perf_samples.append(
            _sp_perf_round(_SP_PERFORMANCE_REQUESTS, _sp_repetition)
        )

    _sp_success_median = _sp_median([row["success"] for row in _sp_perf_samples])
    _sp_fail_median = _sp_median([row["fail"] for row in _sp_perf_samples])
    _sp_p50_values = [
        row["p50_latency_s"] for row in _sp_perf_samples
        if row["p50_latency_s"] is not None
    ]
    _sp_p95_values = [
        row["p95_latency_s"] for row in _sp_perf_samples
        if row["p95_latency_s"] is not None
    ]
    _sp_performance = {
        "version": 2,
        "transport": "http",
        "requests": _SP_PERFORMANCE_REQUESTS,
        "concurrency": _SP_PERFORMANCE_CONCURRENCY,
        "success": int(_sp_success_median),
        "fail": int(_sp_fail_median),
        "success_rate": round(_sp_median(
            [row["success_rate"] for row in _sp_perf_samples]
        ), 4),
        "throughput_rps": round(_sp_median(
            [row["throughput_rps"] for row in _sp_perf_samples]
        ), 4),
        "wall_s": round(_sp_median(
            [row["wall_s"] for row in _sp_perf_samples]
        ), 4),
        "p50_latency_s": (
            round(_sp_median(_sp_p50_values), 4) if _sp_p50_values else None
        ),
        "p95_latency_s": (
            round(_sp_median(_sp_p95_values), 4) if _sp_p95_values else None
        ),
        "errors": [
            dict(error, repetition=sample["repetition"])
            for sample in _sp_perf_samples
            for error in sample["errors"]
        ],
        "summary_method": "median",
        "protocol": {
            "requests_per_run": _SP_PERFORMANCE_REQUESTS,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "warmup_requests": _SP_PERFORMANCE_WARMUP_REQUESTS,
            "repetitions": _SP_PERFORMANCE_REPETITIONS,
        },
        "samples": _sp_perf_samples,
    }
    if _sp_warmup is not None:
        _sp_performance["warmup"] = _sp_warmup
    _sp_print(
        "[성능] closed-loop 중앙값 · "
        + str(_SP_PERFORMANCE_REQUESTS) + "요청 × "
        + str(_SP_PERFORMANCE_REPETITIONS) + "회 · 동시성 "
        + str(_SP_PERFORMANCE_CONCURRENCY) + " · 대표 성공 "
        + str(_sp_performance["success"]) + " · "
        + str(_sp_performance["throughput_rps"]) + " req/s · p95 "
        + str(_sp_performance["p95_latency_s"]) + "s",
        flush=True,
    )

_sp_stop_http_server()

_sp_submission = {"team": _SP_TEAM, "answers": _sp_answers}
if _sp_doc_warnings or _sp_timeouts or _sp_env_warnings or _sp_performance:
    _sp_submission["meta"] = {"doc_name_violations": _sp_doc_warnings,
                              "timeout_qids": _sp_timeouts,
                              "env_warnings": _sp_env_warnings,
                              "transport": _SP_TRANSPORT}
    if _sp_performance:
        _sp_submission["meta"]["performance"] = _sp_performance
_sp_text = _sp_json.dumps(_sp_submission, ensure_ascii=False, indent=2, default=str)
with _sp_open(_SP_OUTPUT_PATH, "w", encoding="utf-8") as _sp_f:
    _sp_f.write(_sp_text)

_sp_print("[완료] " + str(len(_sp_answers)) + "문항 저장: " + _SP_OUTPUT_PATH
      + "  (총 " + str(round(_sp_time.time() - _sp_t0, 1)) + "s)", flush=True)
if _sp_errors:
    _sp_print("[경고] 실패 문항 " + str(len(_sp_errors)) + "건: "
          + ", ".join(_e["qid"] for _e in _sp_errors), flush=True)
if _sp_doc_warnings:
    _sp_print("[경고] 규정 ④ 위반 — 허용 목록 밖 문서명 " + str(len(_sp_doc_warnings)) + "건: "
          + ", ".join(sorted(set(_w["doc"] for _w in _sp_doc_warnings)))
          + "  → 해당 항목은 검색 점수가 0으로 채점됩니다. 허용(띄어쓰기 차이는 무관): "
          + ", ".join(_SP_ALLOWED_DOCS), flush=True)

if _SP_AUTO_DOWNLOAD:
    try:
        from google.colab import files as _sp_files
        _sp_files.download(_SP_OUTPUT_PATH)
        _sp_print("[다운로드] 브라우저 다운로드를 시작했습니다: " + _SP_OUTPUT_PATH, flush=True)
    except Exception as _sp_dl_exc:
        _sp_print("[다운로드] 자동 다운로드 실패(" + type(_sp_dl_exc).__name__ + ": " + str(_sp_dl_exc)
                  + ") — 좌측 파일 탭에서 " + _SP_OUTPUT_PATH + " 를 직접 내려받으세요.", flush=True)

_sp_print("SUBMISSION_RUNNER_DONE " + _sp_json.dumps(
    {"team": _SP_TEAM, "output_path": _SP_OUTPUT_PATH, "n_answers": len(_sp_answers),
     "n_errors": len(_sp_errors), "failed_qids": [_e["qid"] for _e in _sp_errors],
     "n_doc_violations": len(_sp_doc_warnings), "timeout_qids": _sp_timeouts,
     "env_warnings": _sp_env_warnings, "transport": _SP_TRANSPORT,
     "performance": _sp_performance},
    ensure_ascii=False), flush=True)


[서버] FastAPI /health 준비 완료: http://127.0.0.1:8765/health

========== 공개 10문항 실행 · 9팀 ==========
[01/10] P01 실행 중 ...
      (11.6s)
[02/10] P02 실행 중 ...
      (9.9s)
[03/10] P03 실행 중 ...
      (15.8s)
[04/10] P04 실행 중 ...
      (10.6s)
[05/10] P05 실행 중 ...
      (9.5s)
[06/10] P06 실행 중 ...
      (6.9s)
[07/10] P07 실행 중 ...
      (15.4s)
[08/10] P08 실행 중 ...
      (6.8s)
[09/10] P09 실행 중 ...
      (10.5s)
[10/10] P10 실행 중 ...
      (12.3s)
[성능] 워밍업 2요청 실행 중 ...
[성능] 측정 1/3 실행 중 ...
[성능] 측정 2/3 실행 중 ...
[성능] 측정 3/3 실행 중 ...
[성능] closed-loop 중앙값 · 12요청 × 3회 · 동시성 2 · 대표 성공 12 · 0.1064 req/s · p95 22.7047s
[완료] 10문항 저장: /content/answers_public_9.json  (총 464.1s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[다운로드] 브라우저 다운로드를 시작했습니다: /content/answers_public_9.json
SUBMISSION_RUNNER_DONE {"team": "9", "output_path": "/content/answers_public_9.json", "n_answers": 10, "n_errors": 0, "failed_qids": [], "n_doc_violations": 0, "timeout_qids": [], "env_warnings": [], "transport": "http", "performance": {"version": 2, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.1064, "wall_s": 112.7673, "p50_latency_s": 17.796, "p95_latency_s": 22.7047, "errors": [], "summary_method": "median", "protocol": {"requests_per_run": 12, "concurrency": 2, "warmup_requests": 2, "repetitions": 3}, "samples": [{"repetition": 1, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.1052, "wall_s": 114.0415, "p50_latency_s": 17.796, "p95_latency_s": 22.7449, "errors": []}, {"repetition": 2, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rat

In [3]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  평가 셀 — 1번 셀 실행 후 임시 셀에서 실행. 제출 전 반드시 삭제한다.
# =====================================================================================
#  P01~P10 : 운영진 gold_questions_public10.json 원문·정답 조항·key_facts 그대로 내장
#  H01~H18 : 공개 문항이 쓰지 않은 60개 조항에서 직접 만든 자체 검증 세트.
#            비공개 30문항에 대한 일반화 성능을 가늠하는 용도이므로 이쪽을 더 중요하게 본다.
#            H 키팩트는 자체 제작이며 공식 점수가 아닌 일반화 진단용이다.
# =====================================================================================

import json
import re
import time
import unicodedata

QUESTIONS = json.loads(r"""
[
 {
  "id": "P01",
  "ptype": "basic",
  "q": "사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있으며, 다른 사람과 공유하는 것은 허용되나요?",
  "gold": [
   [
    "카카오계정 약관",
    10
   ]
  ],
  "facts": [
   "사업자/단체 카카오계정은 계정 정보에 등록된 담당자 1인만 이용할 수 있습니다.",
   "이를 다른 사람에게 공유하는 것은 금지됩니다."
  ]
 },
 {
  "id": "P02",
  "ptype": "exact",
  "q": "회사가 예측하거나 통제할 수 없는 사유로 서비스가 중단된 경우, 복구가 몇 시간 이상 지연되면 회사는 공지사항에 게시하여 알리나요?",
  "gold": [
   [
    "카카오계정 약관",
    8
   ],
   [
    "카카오 통합서비스약관",
    7
   ],
   [
    "카카오 통합 약관",
    13
   ]
  ],
  "facts": [
   "2시간 이상 복구가 지연되는 경우 카카오 고객센터 공지사항 등에 게시하여 알려 드립니다.",
   "회사는 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력합니다."
  ]
 },
 {
  "id": "P03",
  "ptype": "basic",
  "q": "카카오계정 약관에서 회사가 개별 서비스와 연동하여 카카오계정에서 제공한다고 열거한 '카카오계정 서비스'의 내용 5가지는 각각 무엇인가요?",
  "gold": [
   [
    "카카오계정 약관",
    7
   ]
  ],
  "facts": [
   "카카오계정 서비스의 내용은 통합로그인, SSO(Single Sign On), 카카오계정 정보 통합 관리, 사업자/단체 카카오계정, 기타 회사가 제공하는 서비스입니다.",
   "통합로그인은 카카오계정이 적용된 개별 서비스에서 하나의 카카오계정과 비밀번호로 로그인할 수 있는 통합 회원 인증 서비스입니다.",
   "SSO(Single Sign On)는 웹브라우저나 특정 모바일 기기에서 카카오계정 1회 로그인으로 이용 중인 개별 서비스간 추가 로그인 없이 자동 접속하는 서비스입니다.",
   "카카오계정 정보 통합 관리는 개별 서비스 이용을 위해 카카오계정 정보를 통합 관리하며, 개별 서비스의 유형에 따라 전문기관을 통한 실명확인 및 본인인증을 요청하고 이를 카카오계정 정보로 저장합니다.",
   "사업자/단체 카카오계정은 사업자/단체 명의로 카카오 서비스를 이용하기 위해 만들어진 카카오계정으로서 해당 사업자/단체의 책임 하에 권한을 위임받은 담당자가 이용, 관리할 수 있는 계정 서비스입니다."
  ]
 },
 {
  "id": "P04",
  "ptype": "basic",
  "q": "회사가 위치기반서비스의 이용을 제한하거나 중지한 때에는 이용자에게 무엇을 어떤 방법으로 알리나요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    6
   ]
  ],
  "facts": [
   "회사가 서비스 이용을 제한하거나 중지한 때에는 그 사유 및 제한기간 등을 알립니다.",
   "그 사유 및 제한기간 등은 회사 홈페이지 등을 통해 사전 공지하거나 이용자에게 통지합니다."
  ]
 },
 {
  "id": "P05",
  "ptype": "exact",
  "q": "회사가 위치정보 수집·이용·제공사실 확인자료를 기록·보존하는 근거는 위치정보의 보호 및 이용 등에 관한 법률 제 몇 조 제 몇 항이며, 그 자료는 어디에 기록되어 몇 개월간 보관되나요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    8
   ]
  ],
  "facts": [
   "회사는 위치정보의 보호 및 이용 등에 관한 법률 제16조 제2항에 근거합니다.",
   "위치정보 수집·이용·제공사실 확인자료를 위치정보시스템에 자동으로 기록·보존합니다.",
   "해당 자료는 6개월간 보관합니다."
  ]
 },
 {
  "id": "P06",
  "ptype": "trap",
  "q": "카카오계정이 없는 사람이 통합서비스에 가입하려면 무엇을 먼저 해야 하며, 통합서비스 이용계약은 동의·확인·승낙의 어떤 순서로 체결되나요?",
  "gold": [
   [
    "카카오 통합서비스약관",
    4
   ]
  ],
  "facts": [
   "통합서비스에 가입하기 위해서는 카카오계정이 필요하며, 카카오계정이 없는 경우 카카오계정을 먼저 생성하여야 합니다.",
   "통합서비스 이용계약은 여러분이 본 약관의 내용에 동의한 후 회사가 여러분의 카카오계정 정보 등을 확인한 후 승낙함으로써 체결됩니다."
  ]
 },
 {
  "id": "P07",
  "ptype": "basic",
  "q": "서비스 명칭에 '카카오'가 사용되더라도 카카오 통합서비스약관의 '통합서비스'에 포함되지 않는 서비스는 누가 제공하는 서비스이며, 약관은 그 예로 무엇을 들고 있나요?",
  "gold": [
   [
    "카카오 통합서비스약관",
    1
   ]
  ],
  "facts": [
   "서비스 명칭에 '카카오'가 사용되더라도 회사가 아닌 카카오 계열사에서 제공하는 서비스는 본 약관의 통합서비스에 포함되지 않습니다.",
   "그 예는 ㈜카카오모빌리티가 제공하는 카카오 T 택시 서비스입니다.",
   "여러분은 회사가 아닌 계열사를 포함한 제3자가 제공하는 서비스에 가입되지는 않습니다."
  ]
 },
 {
  "id": "P08",
  "ptype": "trap",
  "q": "카카오 통합 약관과 세부지침(회사가 정한 서비스의 개별 이용약관·운영정책·규칙 등)의 내용이 충돌하는 경우, 본 약관이 세부지침보다 우선하여 적용되나요?",
  "gold": [
   [
    "카카오 통합 약관",
    3
   ]
  ],
  "facts": [
   "아니오. 본 약관과 세부지침의 내용이 충돌할 경우 세부지침에 따릅니다.",
   "본 약관에 규정되지 않은 사항에 대해서는 관련법령 또는 회사가 정한 서비스의 개별 이용약관, 운영정책 및 규칙 등(세부지침)의 규정에 따릅니다."
  ]
 },
 {
  "id": "P09",
  "ptype": "trap",
  "q": "이용자가 서비스 사용을 중단하거나 카카오계정 및 Daum 아이디를 탈퇴한 이후, 게시물에 관하여 회사에 부여한 라이선스의 효력은 어떻게 되나요?",
  "gold": [
   [
    "카카오 통합 약관",
    10
   ]
  ],
  "facts": [
   "본 라이선스는 여러분이 서비스의 사용을 중단하거나 카카오계정 및/또는 Daum 아이디를 탈퇴한 후에도 존속하게 됩니다.",
   "여러분이 회사에게 제공하는 라이선스는 전 세계적이고 영구적인 라이선스입니다."
  ]
 },
 {
  "id": "P10",
  "ptype": "basic",
  "q": "8세 이하의 아동 등의 생명 또는 신체 보호를 위해 보호의무자가 개인위치정보의 이용 또는 제공에 동의하려면 어떤 서류에 무엇을 첨부하여 어디에 제출해야 하며, 그 동의는 어떤 효력을 갖나요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    12
   ]
  ],
  "facts": [
   "8세 이하의 아동 등의 생명 또는 신체의 보호를 위하여 개인위치정보의 이용 또는 제공에 동의를 하고자 하는 보호의무자는 서면동의서에 보호의무자임을 증명하는 서면을 첨부하여 회사에 제출하여야 합니다.",
   "보호의무자가 개인위치정보의 이용 또는 제공에 동의하는 경우에는 본인의 동의가 있는 것으로 봅니다.",
   "보호의무자는 이 경우 이용자의 권리를 모두 가집니다."
  ]
 },
 {
  "id": "H01",
  "ptype": "exact",
  "q": "카카오 위치정보 서비스의 이용요금은 얼마이며, 데이터 통신료는 어떻게 처리되나요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    5
   ]
  ],
  "facts": [
   "무료입니다",
   "데이터 통신료는 별도",
   "각 이동통신사의 정책에 따릅니다"
  ]
 },
 {
  "id": "H02",
  "ptype": "basic",
  "q": "카카오 위치정보 이용약관에 기재된 회사의 주소와 위치정보관리책임자의 성명은 각각 무엇인가요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    16
   ]
  ],
  "facts": [
   "제주특별자치도 제주시 첨단로 242 (영평동)",
   "김연지"
  ]
 },
 {
  "id": "H03",
  "ptype": "exact",
  "q": "인증서는 명의자 당 몇 개의 카카오계정에서 몇 대의 기기에 발급되며, 다른 기기에서 재발급하면 기존 인증서는 어떻게 되나요?",
  "gold": [
   [
    "카카오 통합 약관",
    17
   ]
  ],
  "facts": [
   "명의자 당 1개의 카카오계정에서 1대의 기기에만 발급됩니다",
   "기존에 발급받은 인증서는 자동 폐지됩니다"
  ]
 },
 {
  "id": "H04",
  "ptype": "basic",
  "q": "회사가 인증회원에게 제공한다고 열거한 인증서비스의 종류 네 가지는 각각 무엇인가요?",
  "gold": [
   [
    "카카오 통합 약관",
    17
   ]
  ],
  "facts": [
   "전자서명생성정보 및 인증서 발급",
   "전자서명 및 인증서를 활용한 각종 서비스",
   "이용기관 로그인 및 신원확인을 위한 간편인증",
   "기타 전자서명인증업무 운영준칙에서 정하는 서비스"
  ]
 },
 {
  "id": "H05",
  "ptype": "trap",
  "q": "카카오 통합서비스약관 제10조의 내용과 개별 서비스에 적용되는 유료서비스 약관의 내용이 충돌하는 경우, 통합서비스약관이 우선 적용되나요?",
  "gold": [
   [
    "카카오 통합서비스약관",
    10
   ]
  ],
  "facts": [
   "개별 서비스에 적용되는 유료서비스 약관의 규정에 따릅니다"
  ]
 },
 {
  "id": "H06",
  "ptype": "exact",
  "q": "회사는 환불 의무가 발생한 날로부터 며칠 이내에 환불을 진행하며, 환불이 지연되는 경우 지연이자율은 얼마인가요?",
  "gold": [
   [
    "카카오 통합 약관",
    11
   ]
  ],
  "facts": [
   "3영업일 이내에 환불을 진행하며",
   "지연이자율은 연리 11%"
  ]
 },
 {
  "id": "H07",
  "ptype": "exact",
  "q": "이용요금에 관한 이의는 언제까지 제기해야 하나요?",
  "gold": [
   [
    "카카오 통합 약관",
    11
   ]
  ],
  "facts": [
   "사유 발생을 안 날로부터 1월",
   "발생한 날로부터 3월 이내에 제기"
  ]
 },
 {
  "id": "H08",
  "ptype": "basic",
  "q": "14세 미만 아동의 개인위치정보를 고지한 범위를 넘어 이용하거나 제3자에게 제공할 때 법정대리인 동의가 필요 없는 예외 두 가지는 무엇인가요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    11
   ]
  ],
  "facts": [
   "요금정산을 위하여 위치정보 이용, 제공사실 확인자료가 필요한 경우",
   "통계작성, 학술연구 또는 시장조사를 위하여 특정 개인을 알아볼 수 없는 형태로 가공하여 제공하는 경우"
  ]
 },
 {
  "id": "H09",
  "ptype": "exact",
  "q": "카카오계정 약관에서 서비스 이용자 전체에 대한 공지는 며칠 이상 게시해야 효력이 발생하나요?",
  "gold": [
   [
    "카카오계정 약관",
    14
   ]
  ],
  "facts": [
   "칠(7)일 이상",
   "게시함으로써 효력이 발생합니다"
  ]
 },
 {
  "id": "H10",
  "ptype": "basic",
  "q": "카카오계정 로그인이나 접속 기록이 없는 경우 회사는 어떤 조치를 할 수 있으며, 자세한 사항은 어떤 정책을 참고하라고 안내하나요?",
  "gold": [
   [
    "카카오계정 약관",
    15
   ]
  ],
  "facts": [
   "카카오계정 정보를 파기하거나 분리 보관할 수 있으며",
   "서비스 장기 미이용 처리 정책을 참고"
  ]
 },
 {
  "id": "H11",
  "ptype": "exact",
  "q": "통합서비스에서 신고된 이용자의 음성정보는 신고 접수시부터 얼마 동안 보관된 후 어떻게 되나요?",
  "gold": [
   [
    "카카오 통합서비스약관",
    12
   ]
  ],
  "facts": [
   "3년간",
   "보관 후 파기합니다"
  ]
 },
 {
  "id": "H12",
  "ptype": "basic",
  "q": "카카오 통합 약관에서 8세 이하의 아동 등의 보호의무자가 서면으로 동의하는 경우 어떤 효력이 있으며 보호의무자는 어떤 권리를 갖나요?",
  "gold": [
   [
    "카카오 통합 약관",
    16
   ]
  ],
  "facts": [
   "해당 본인의 동의가 있는 것으로 보며",
   "개인위치정보주체의 권리를 모두 행사할 수 있습니다"
  ]
 },
 {
  "id": "H13",
  "ptype": "basic",
  "q": "카카오계정 정보를 적시에 수정하지 않아 문제가 발생한 경우 회사는 어떤 조건에서 책임을 지지 않나요?",
  "gold": [
   [
    "카카오계정 약관",
    9
   ]
  ],
  "facts": [
   "회사의 고의 또는 과실이 없는 한 회사는 책임을 부담하지 아니합니다"
  ]
 },
 {
  "id": "H14",
  "ptype": "basic",
  "q": "카카오 위치정보 이용약관이 열거한 위치정보 서비스 다섯 가지는 각각 무엇인가요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    4
   ]
  ],
  "facts": [
   "검색결과 제공 및 콘텐츠 추천",
   "생활편의 서비스 제공",
   "위치 기반 콘텐츠 분류(Geo Tagging)",
   "위치기반 소셜 서비스 제공",
   "위치기반 광고"
  ]
 },
 {
  "id": "H15",
  "ptype": "trap",
  "q": "카카오 통합서비스약관은 이용계약이 해지되면 모든 조항의 적용이 즉시 끝나나요?",
  "gold": [
   [
    "카카오 통합서비스약관",
    2
   ]
  ],
  "facts": [
   "일부 조항은 이용계약의 해지 후에도 유효하게 적용될 수 있습니다"
  ]
 },
 {
  "id": "H16",
  "ptype": "basic",
  "q": "카카오계정 이용 신청은 어떤 방식으로 이루어지며, 카카오계정 이용계약은 어떤 순서로 체결되나요?",
  "gold": [
   [
    "카카오계정 약관",
    5
   ]
  ],
  "facts": [
   "카카오계정 정보에 일정 정보를 입력하는 방식",
   "본 약관의 내용에 동의한 후",
   "입력된 일정 정보를 인증한 후 가입을 승낙함으로써 체결됩니다"
  ]
 },
 {
  "id": "H17",
  "ptype": "exact",
  "q": "통합서비스에서 청소년유해매체물을 이용하려면 몇 세 이상이어야 하며 어떤 절차를 거쳐야 하나요?",
  "gold": [
   [
    "카카오 통합서비스약관",
    12
   ]
  ],
  "facts": [
   "만 19세 이상이어야 하며",
   "실명인증을 통해 본인 및 연령 인증을 받으셔야 합니다"
  ]
 },
 {
  "id": "H18",
  "ptype": "exact",
  "q": "카카오 위치정보 이용약관이 변경될 때 회사는 적용일자 며칠 전에 공지하며, 이용자 권리의 중대한 변경인 경우에는 며칠 전에 개별 고지하나요?",
  "gold": [
   [
    "카카오 위치정보 이용약관",
    2
   ]
  ],
  "facts": [
   "적용일자 최소 15일 전",
   "적용일 최소 30일 전"
  ]
 }
]
""")

PUBLIC_GOLD_DIGEST = "a778ca719726b80517b19107e81a4c3616ccf3ec4050b41965d9832b65eb03da"




from collections import Counter


def _n(value):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(value))).lower()


_TOKEN_RE = re.compile(r"[0-9]+(?:\.[0-9]+)?|[A-Za-z]+|[가-힣]+")
_JOSA = (
    "으로써", "에게서", "에서는", "이라고", "으로는", "에서", "에게", "으로", "라고",
    "까지", "부터", "보다", "처럼", "이나", "마다", "조차", "한테", "와의", "과의",
    "의", "가", "이", "은", "는", "을", "를", "와", "과", "로", "도", "만", "에",
)
FACT_RECALL_THRESHOLD = 0.55


def _tokens(value):
    tokens = []
    for token in _TOKEN_RE.findall(unicodedata.normalize("NFKC", str(value)).lower()):
        if re.fullmatch(r"[가-힣]+", token) and len(token) >= 3:
            for suffix in _JOSA:
                if token.endswith(suffix) and len(token) - len(suffix) >= 2:
                    token = token[:-len(suffix)]
                    break
        tokens.append(token)
    return tokens


def _overlap(gold_text, answer):
    gold_counts = Counter(_tokens(gold_text))
    answer_counts = Counter(_tokens(answer))
    return gold_counts, answer_counts, sum((gold_counts & answer_counts).values())


def _fact_recall(fact, answer):
    gold_counts, _, overlap = _overlap(fact, answer)
    return overlap / sum(gold_counts.values()) if gold_counts else 1.0


def _token_f1(gold_text, answer):
    gold_counts, answer_counts, overlap = _overlap(gold_text, answer)
    if not gold_counts or not answer_counts or not overlap:
        return 0.0
    precision = overlap / sum(answer_counts.values())
    recall = overlap / sum(gold_counts.values())
    return 2 * precision * recall / (precision + recall)


def _validate_output(output):
    if not isinstance(output, dict):
        raise TypeError("answer_question은 dict를 반환해야 합니다")
    answer = output.get("answer")
    retrieved = output.get("retrieved")
    if not isinstance(answer, str):
        raise TypeError("answer는 문자열이어야 합니다")
    if not isinstance(retrieved, list) or not 1 <= len(retrieved) <= 4:
        raise ValueError("retrieved는 1~4개의 [문서명, 조번호] 목록이어야 합니다")
    pairs = []
    for pair in retrieved:
        if not isinstance(pair, (list, tuple)) or len(pair) != 2:
            raise TypeError("retrieved 항목은 [문서명, 조번호] 형식이어야 합니다")
        document, article = pair
        if not isinstance(document, str) or isinstance(article, bool) or not isinstance(article, int):
            raise TypeError("문서명은 문자열, 조번호는 정수여야 합니다")
        pairs.append((_n(document), article))
    return answer, pairs


def _looks_cut(answer):
    text = answer.rstrip()
    if not text or text.endswith(("다.", "요.", ".", "!", "?", "다", "요", ")", "]", "”", "’")):
        return False
    return bool(re.search(r"(?:[,;:]|이|가|은|는|을|를|의|에|에서|에게|으로|하여|하고|또는|및)$", text))


def _has_repetition(answer):
    sentences = [_n(s) for s in re.split(r"(?<=다\.)\s*|\n+", answer) if len(_n(s)) >= 15]
    return len(sentences) != len(set(sentences))


rows = []
print("=" * 82)
for item in QUESTIONS:
    started = time.perf_counter()
    try:
        answer, got = _validate_output(answer_question(item["q"]))  # noqa: F821
        error = None
    except Exception as exc:
        answer, got, error = "", [], f"{type(exc).__name__}: {exc}"
    seconds = time.perf_counter() - started

    gold = [(_n(document), int(article)) for document, article in item["gold"]]
    hit = any(g in got for g in gold)
    rr = next((1.0 / (rank + 1) for rank, pair in enumerate(got) if pair in gold), 0.0)
    fact_recalls = [_fact_recall(fact, answer) for fact in item["facts"]]
    weak_facts = [
        (fact, score) for fact, score in zip(item["facts"], fact_recalls)
        if score < FACT_RECALL_THRESHOLD
    ]
    keyfact_f1 = _token_f1(" ".join(item["facts"]), answer)
    cut_suspected = _looks_cut(answer)
    repeated = _has_repetition(answer)

    rows.append(dict(
        qid=item["id"], ptype=item["ptype"], group=item["id"][0], hit=hit, rr=rr,
        keyfact_f1=keyfact_f1, weak_facts=weak_facts, cut=cut_suspected,
        rep=repeated, sec=seconds, err=error, ans=answer,
    ))

    ok = hit and not weak_facts and not cut_suspected and not repeated and not error
    print(
        f"{'OK ' if ok else '!! '}{item['id']} [{item['ptype']:5s}] "
        f"Hit@4 {int(hit)} RR {rr:.2f} | 키팩트 F1 {keyfact_f1:.3f} | "
        f"{seconds:5.1f}s | {len(answer):>4}자"
    )
    if error:
        print(f"      오류: {error}")
    for fact, score in weak_facts:
        print(f"      약한 키팩트 {score:.0%}: {fact[:80]}")
    if cut_suspected:
        print("      문장 종결 의심: 실제 답변이 잘렸는지 전문을 확인하세요")
    if repeated:
        print("      동일 문장 반복: repetition_penalty 또는 프롬프트를 확인하세요")


def summarize(name, selected):
    if not selected:
        return
    count = len(selected)
    latency = sorted(row["sec"] for row in selected)
    mean_mrr = sum(row["rr"] for row in selected) / count
    mean_f1 = sum(row["keyfact_f1"] for row in selected) / count
    print(f"\n[{name}] {count}문항")
    print(f"  검색  Hit@4 {sum(row['hit'] for row in selected)/count:.3f}   MRR {mean_mrr:.3f}")
    print(
        f"  생성  키팩트 토큰 F1 프록시 {mean_f1:.3f}   "
        f"종결 의심 {sum(row['cut'] for row in selected)}   반복 {sum(row['rep'] for row in selected)}"
    )
    print(
        f"  지연  p50 {latency[count//2]:.1f}s   최대 {latency[-1]:.1f}s   "
        f"평균 길이 {sum(len(row['ans']) for row in selected)//count}자"
    )
    return mean_mrr, mean_f1


print("=" * 82)
public_metrics = summarize("공개 P — 공식 gold_questions_public10 기반", [r for r in rows if r["group"] == "P"])
summarize("검증 H — 자체 제작, 비공식", [r for r in rows if r["group"] == "H"])
summarize("전체 — 참고용", rows)
for ptype in ("basic", "exact", "trap"):
    selected = [row for row in rows if row["ptype"] == ptype]
    if selected:
        print(
            f"  유형 {ptype:5s} 키팩트 F1 {sum(row['keyfact_f1'] for row in selected)/len(selected):.3f}  "
            f"MRR {sum(row['rr'] for row in selected)/len(selected):.3f}"
        )

if public_metrics:
    public_mrr, public_f1 = public_metrics
    print(
        f"\n공개 정량 프록시: MRR {public_mrr*20:.3f}/20 + "
        f"키팩트 F1 {public_f1*30:.3f}/30 = {public_mrr*20 + public_f1*30:.3f}/50"
    )
    print("통합 LLM 평가 50점은 이 로컬 셀에서 계산하지 않습니다.")

print("\n판단 기준")
print("  · Hit@4 < 1.0            -> 검색 결과의 정답 조항 포함 여부를 먼저 확인")
print("  · MRR < 1.0              -> 정답 조항을 더 높은 순위로 올리도록 검색 조정")
print("  · 키팩트 F1이 낮음       -> 약한 키팩트와 불필요한 답변 내용을 함께 확인")
print("  · 종결 의심/반복 발생    -> 답변 전문을 확인한 뒤 생성 길이·프롬프트 조정")
print("  · H 결과                 -> 자체 문항 기반 일반화 참고치이며 공식 점수가 아님")

print("\n" + "=" * 82)
print("실패·주의 문항 답변 전문")
for row in rows:
    if not (row["hit"] and not row["weak_facts"] and not row["cut"] and not row["rep"] and not row["err"]):
        print(f"\n[{row['qid']}] {row['ans'][:1200]}")


OK P01 [basic] Hit@4 1 RR 1.00 | 키팩트 F1 0.944 |  11.7s |   69자
!! P02 [exact] Hit@4 1 RR 1.00 | 키팩트 F1 0.585 |  10.8s |   68자
      약한 키팩트 18%: 회사는 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력합니다.
!! P03 [basic] Hit@4 1 RR 1.00 | 키팩트 F1 0.262 |  15.0s |   84자
      약한 키팩트 33%: 통합로그인은 카카오계정이 적용된 개별 서비스에서 하나의 카카오계정과 비밀번호로 로그인할 수 있는 통합 회원 인증 서비스입니다.
      약한 키팩트 26%: SSO(Single Sign On)는 웹브라우저나 특정 모바일 기기에서 카카오계정 1회 로그인으로 이용 중인 개별 서비스간 추가 로그인 없이 자
      약한 키팩트 23%: 카카오계정 정보 통합 관리는 개별 서비스 이용을 위해 카카오계정 정보를 통합 관리하며, 개별 서비스의 유형에 따라 전문기관을 통한 실명확인 및 
      약한 키팩트 15%: 사업자/단체 카카오계정은 사업자/단체 명의로 카카오 서비스를 이용하기 위해 만들어진 카카오계정으로서 해당 사업자/단체의 책임 하에 권한을 위임받
OK P04 [basic] Hit@4 1 RR 1.00 | 키팩트 F1 0.818 |   9.4s |   81자
OK P05 [exact] Hit@4 1 RR 1.00 | 키팩트 F1 0.862 |   9.6s |  106자
!! P06 [trap ] Hit@4 1 RR 1.00 | 키팩트 F1 0.300 |   6.9s |   52자
      약한 키팩트 33%: 통합서비스에 가입하기 위해서는 카카오계정이 필요하며, 카카오계정이 없는 경우 카카오계정을 먼저 생성하여야 합니다.
      약한 키팩트 24%: 통합서비스 이용계약은 여러분이 본 약관의 내용에 동의한 후 회사가 여러분의 카카오계정 정보 등을 확인한 후 승낙함으로써 체결됩니